# Notebook 6 — Base maestra v3 + preparación del modelo final (PU learning + redes)

**Objetivo de esta etapa:** dejar una sola base lista para el modelo final:
1. Recuperar el entorno completo después del reinicio del kernel.
2. Integrar a `base_maestra_v2` todas las variables relacionales del notebook 4 (dominancia, win rate,
   competidor habitual, Benford, entrada a mercado) + capital suscrito de la Superintendencia → `base_maestra_v3`.
3. Diagnosticar las etiquetas (fantasmas) y revisar una posible fuga por empresa (el mismo proveedor en
   entrenamiento y prueba).

Regla de todo el proyecto: **cero fuga temporal**. Las tablas satélite ya vienen calculadas por
`anio_aplicable` = año del proceso, usando solo información de años anteriores; por eso el cruce es
`anio del proceso = anio_aplicable`.


## Celda 1 — Recuperar el entorno (librerías, carpeta, vistas sobre los parquet)

In [ ]:
import os, sys, time
import numpy as np
import pandas as pd
import duckdb

os.chdir(r'C:\SERCOP_FINAL')
R = '03_resultados'
print('Carpeta de trabajo:', os.getcwd())

con = duckdb.connect()
con.execute("PRAGMA threads=8")

# Se usan VISTAS (no tablas) sobre los parquet: no ocupan RAM y evitan el problema de
# tablas viejas en el catálogo de DuckDB que ya nos pasó con directorio_ref.
fuentes = {
    'bm':          'base_maestra_v2.parquet',
    'dominancia':  'perfil_dominancia_expandido.parquet',
    'winrate':     'winrate_expandido.parquet',
    'competidor':  'competidor_habitual.parquet',
    'benford':     'benford_expandido.parquet',
    'entrada':     'entrada_mercado.parquet',
    'directorio':  'directorio_companias.parquet',
}
for vista, archivo in fuentes.items():
    ruta = os.path.join(R, archivo).replace('\\', '/')
    if not os.path.exists(ruta):
        print(f'!! NO EXISTE: {ruta}')
        continue
    con.execute(f"CREATE OR REPLACE VIEW {vista} AS SELECT * FROM read_parquet('{ruta}')")
    n = con.execute(f"SELECT COUNT(*) FROM {vista}").fetchone()[0]
    cols = [c[0] for c in con.execute(f"DESCRIBE {vista}").fetchall()]
    print(f'\n[{vista}] {n:,} filas | {len(cols)} columnas')
    print('   ', cols)


**Pásame la salida completa de la Celda 1** (sobre todo la lista de columnas de `bm`), antes de seguir,
por si algún nombre de columna no coincide con lo que asume la Celda 2.

## Celda 2 — Construir `base_maestra_v3` (todo en DuckDB, sin cargar 2.7M filas a pandas)

In [ ]:
con.execute(f"""CREATE OR REPLACE VIEW winrate AS
    SELECT oferente_ruc AS proveedor_ruc, * EXCLUDE (oferente_ruc)
    FROM read_parquet('{R}/winrate_expandido.parquet')""")

In [ ]:
bm_cols = [c[0] for c in con.execute("DESCRIBE bm").fetchall()]

# --- 1) Detectar la columna de año del proceso ---
anio_expr = None
for c in ['anio_proceso', 'anio', 'anio_release', 'year']:
    if c in bm_cols:
        anio_expr = f'b."{c}"'; break
if anio_expr is None:
    for c in ['fecha_release', 'fecha_publicacion', 'date', 'fecha']:
        if c in bm_cols:
            anio_expr = f'EXTRACT(year FROM CAST(b."{c}" AS TIMESTAMP))'; break
assert anio_expr is not None, 'No encontré columna de año/fecha: revisar nombres en la Celda 1'
assert 'ocid' in bm_cols and 'proveedor_ruc' in bm_cols, 'Faltan ocid / proveedor_ruc en bm'
print('Año del proceso tomado de:', anio_expr)

# --- 2) Verificar que cada tabla satélite sea única por su llave (si no, multiplicaría filas) ---
llaves = {
    'dominancia': ['proveedor_ruc', 'anio_aplicable'],
    'winrate':    ['proveedor_ruc', 'anio_aplicable'],
    'competidor': ['proveedor_ruc', 'anio_aplicable'],
    'benford':    ['proveedor_ruc', 'anio_aplicable'],
    'entrada':    ['ocid', 'proveedor_ruc'],
}
for t, k in llaves.items():
    kk = ', '.join(k)
    dup = con.execute(f"SELECT COUNT(*) FROM (SELECT {kk} FROM {t} GROUP BY {kk} HAVING COUNT(*) > 1)").fetchone()[0]
    print(f'{t}: llaves duplicadas = {dup:,}')
    assert dup == 0, f'{t} tiene llaves duplicadas: hay que deduplicar antes del join'

# --- 3) Lista de columnas nuevas, con prefijo si el nombre ya existe en la base ---
ya = set(bm_cols)
select_nuevas, alias_tabla = [], {'dominancia': 'd', 'winrate': 'w', 'competidor': 'c', 'benford': 'bf', 'entrada': 'e'}
for t, k in llaves.items():
    for col in [c[0] for c in con.execute(f"DESCRIBE {t}").fetchall()]:
        if col in k:
            continue
        nombre = col if col not in ya else f'{t}_{col}'
        select_nuevas.append(f'{alias_tabla[t]}."{col}" AS "{nombre}"')
        ya.add(nombre)

# Capital suscrito del proveedor (Superintendencia). El directorio tiene 1 RUC repetido -> se deduplica.
select_nuevas.append('cap.capital_suscrito AS proveedor_capital_suscrito')

sql = f"""
COPY (
  WITH cap AS (
      SELECT RUC, MAX(TRY_CAST("CAPITAL SUSCRITO" AS DOUBLE)) AS capital_suscrito
      FROM directorio GROUP BY RUC
  )
  SELECT b.*,
         {', '.join(select_nuevas)}
  FROM bm b
  LEFT JOIN dominancia d ON b.proveedor_ruc = d.proveedor_ruc AND {anio_expr} = d.anio_aplicable
  LEFT JOIN winrate    w ON b.proveedor_ruc = w.proveedor_ruc AND {anio_expr} = w.anio_aplicable
  LEFT JOIN competidor c ON b.proveedor_ruc = c.proveedor_ruc AND {anio_expr} = c.anio_aplicable
  LEFT JOIN benford   bf ON b.proveedor_ruc = bf.proveedor_ruc AND {anio_expr} = bf.anio_aplicable
  LEFT JOIN entrada    e ON b.ocid = e.ocid AND b.proveedor_ruc = e.proveedor_ruc
  LEFT JOIN cap          ON b.proveedor_ruc = cap.RUC
) TO '{R}/base_maestra_v3.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
"""
t0 = time.time()
con.execute(sql)
print(f'base_maestra_v3 escrita en {time.time()-t0:.0f} s')

con.execute(f"CREATE OR REPLACE VIEW bm3 AS SELECT * FROM read_parquet('{R}/base_maestra_v3.parquet')")
n2 = con.execute("SELECT COUNT(*) FROM bm").fetchone()[0]
n3 = con.execute("SELECT COUNT(*) FROM bm3").fetchone()[0]
cols3 = [c[0] for c in con.execute("DESCRIBE bm3").fetchall()]
print(f'Filas v2: {n2:,} | Filas v3: {n3:,} | Columnas v3: {len(cols3)}')
assert n2 == n3, 'El número de filas cambió: algún join está duplicando'

# --- 4) % de nulos de las columnas nuevas, por año (esperado: 2015 casi todo nulo, no hay año previo) ---
nuevas = [c for c in cols3 if c not in bm_cols]
print('\nColumnas nuevas:', nuevas)
exprs = ', '.join([f'ROUND(100.0*AVG(CASE WHEN "{c}" IS NULL THEN 1 ELSE 0 END),1) AS "{c}"' for c in nuevas])
anio3 = anio_expr.replace('b.', '')
nulos = con.execute(f"SELECT {anio3} AS anio, {exprs} FROM bm3 GROUP BY 1 ORDER BY 1").df()
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 60)
print('\n% de nulos por año (columnas nuevas):')
print(nulos.T.to_string())


## Celda 3 — Diagnóstico de etiquetas y fuga por empresa

In [ ]:
# ¿Cuántos fantasmas (filas y RUC únicos) hay por año?
etq = con.execute(f"""
    SELECT {anio3} AS anio,
           COUNT(*) AS filas,
           SUM(CASE WHEN proveedor_es_fantasma THEN 1 ELSE 0 END) AS filas_fantasma,
           COUNT(DISTINCT CASE WHEN proveedor_es_fantasma THEN proveedor_ruc END) AS ruc_fantasma
    FROM bm3 GROUP BY 1 ORDER BY 1
""").df()
print(etq.to_string(index=False))
print('\nRUC fantasma únicos en toda la base:',
      con.execute("SELECT COUNT(DISTINCT proveedor_ruc) FROM bm3 WHERE proveedor_es_fantasma").fetchone()[0])

# Fuga por empresa: ¿los fantasmas del test (2018-2019) ya aparecían como fantasma en el train (2015-2017)?
# Si se repiten, el modelo puede estar "reconociendo a la empresa" y no aprendiendo el patrón.
tr = set(con.execute(f"SELECT DISTINCT proveedor_ruc FROM bm3 WHERE proveedor_es_fantasma AND {anio3} BETWEEN 2015 AND 2017").df()['proveedor_ruc'])
te = set(con.execute(f"SELECT DISTINCT proveedor_ruc FROM bm3 WHERE proveedor_es_fantasma AND {anio3} BETWEEN 2018 AND 2019").df()['proveedor_ruc'])
print(f'\nFantasmas únicos train: {len(tr)} | test: {len(te)} | presentes en ambos: {len(tr & te)}')
print(f'Fantasmas del test que son NUEVOS (no vistos en train): {len(te - tr)}')


**Pásame la salida de las Celdas 2 y 3.** Con eso se decide:
- si `base_maestra_v3` queda como base final del modelo (filas sin cambio, nulos esperados);
- cómo se hace la partición del modelo final: además del corte temporal, una partición **por empresa**
  (ningún proveedor en train y test a la vez), como hace el estudio de México (Scientific Reports, 2026).


In [ ]:
# ============ A) Diagnóstico de capital suscrito ============
print(con.execute("""
    SELECT typeof("CAPITAL SUSCRITO") AS tipo_dato, typeof(RUC) AS tipo_ruc,
           MIN(LENGTH(CAST(RUC AS VARCHAR))) AS min_len_ruc, MAX(LENGTH(CAST(RUC AS VARCHAR))) AS max_len_ruc,
           COUNT(*) AS filas
    FROM directorio GROUP BY 1, 2
""").df().to_string(index=False))
print('\nEjemplos de CAPITAL SUSCRITO tal como viene:')
print(con.execute("""SELECT RUC, "CAPITAL SUSCRITO" FROM directorio
                     WHERE "CAPITAL SUSCRITO" IS NOT NULL LIMIT 10""").df().to_string(index=False))
print('\nProveedores de la base que cruzan con el directorio por RUC:',
      con.execute("""SELECT COUNT(DISTINCT b.proveedor_ruc) FROM bm3 b
                     JOIN directorio d ON b.proveedor_ruc = CAST(d.RUC AS VARCHAR)""").fetchone()[0])

# ============ B) Cambio de régimen de publicación por año ============
print('\nCobertura de información competitiva por año (% de filas):')
print(con.execute("""
    SELECT anio_proceso AS anio,
           ROUND(100*AVG(CASE WHEN n_oferentes IS NOT NULL THEN 1 ELSE 0 END),1) AS pct_con_n_oferentes,
           ROUND(100*AVG(CASE WHEN tiene_periodo_competitivo THEN 1 ELSE 0 END),1) AS pct_competitivo,
           ROUND(100*AVG(CASE WHEN proveedor_unico THEN 1 ELSE 0 END),1) AS pct_proveedor_unico,
           ROUND(AVG(n_oferentes),2) AS prom_oferentes
    FROM bm3 GROUP BY 1 ORDER BY 1
""").df().to_string(index=False))

# ============ C) Chequeo de coherencia: entrada_anio_proceso debe ser igual a anio_proceso ============
print('\nFilas donde entrada_anio_proceso != anio_proceso:',
      con.execute("""SELECT COUNT(*) FROM bm3
                     WHERE entrada_anio_proceso IS NOT NULL AND entrada_anio_proceso <> anio_proceso""").fetchone()[0])

# ============ D) ¿Dónde están los 15 fantasmas? (método y provincia) ============
print('\nFantasmas por método de contratación y provincia del proveedor:')
print(con.execute("""
    SELECT metodo_contratacion, provincia_proveedor,
           COUNT(*) AS filas, COUNT(DISTINCT proveedor_ruc) AS ruc
    FROM bm3 WHERE proveedor_es_fantasma
    GROUP BY 1, 2 ORDER BY filas DESC
""").df().to_string(index=False))

In [ ]:
# ============ Arreglar capital suscrito (formato 48.200,00) y completar base_maestra_v3 ============
import os
con.execute(f"""
COPY (
  WITH cap AS (
      SELECT RUC,
             MAX(TRY_CAST(REPLACE(REPLACE(TRIM("CAPITAL SUSCRITO"), '.', ''), ',', '.') AS DOUBLE)) AS capital
      FROM directorio GROUP BY RUC
  )
  SELECT b.* EXCLUDE (proveedor_capital_suscrito, entrada_anio_proceso),
         cap.capital AS proveedor_capital_suscrito,
         -- ratio monto adjudicado / capital (solo sociedades con capital > 0)
         CASE WHEN cap.capital > 0 THEN b.monto_final / cap.capital END AS ratio_monto_capital,
         -- banderas explícitas para nulos estructurales (misma regla de todo el proyecto: no imputar)
         (b.win_rate IS NULL)                    AS sin_historial_como_oferente,
         (b.dominancia_maxima_historica IS NULL) AS sin_pares_recurrentes,
         (b.mad_benford IS NULL)                 AS sin_historial_benford
  FROM bm3 b
  LEFT JOIN cap ON b.proveedor_ruc = cap.RUC
) TO '{R}/base_maestra_v3_tmp.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
""")
con.execute("DROP VIEW IF EXISTS bm3")          # suelta el archivo viejo antes de reemplazarlo
os.replace(f'{R}/base_maestra_v3_tmp.parquet', f'{R}/base_maestra_v3.parquet')
con.execute(f"CREATE OR REPLACE VIEW bm3 AS SELECT * FROM read_parquet('{R}/base_maestra_v3.parquet')")

print(con.execute("""
    SELECT COUNT(*) AS filas,
           ROUND(100*AVG(CASE WHEN proveedor_capital_suscrito IS NOT NULL THEN 1 ELSE 0 END),2) AS pct_con_capital,
           ROUND(100*AVG(CASE WHEN proveedor_es_sociedad_registrada THEN 1 ELSE 0 END),2)        AS pct_sociedad,
           quantile_cont(proveedor_capital_suscrito, 0.5) AS capital_mediano,
           quantile_cont(ratio_monto_capital, 0.5)        AS ratio_mediano,
           quantile_cont(ratio_monto_capital, 0.99)       AS ratio_p99,
           SUM(CASE WHEN ratio_monto_capital > 100 THEN 1 ELSE 0 END) AS filas_monto_100x_capital
    FROM bm3
""").df().T.to_string())
print('Columnas v3:', len(con.execute("DESCRIBE bm3").fetchall()))

In [ ]:
# ============ Validación de red flags al estilo Fazekas ============
# Pregunta: dentro de los procesos competitivos, ¿cada bandera se asocia con que haya UN SOLO oferente?
# Si una bandera no se asocia con proveedor único, no entra (o pesa poco) en el índice de riesgo.
# Se excluye 2015 (año de arranque de la ventana expandida, sin historial).
import numpy as np, pandas as pd
try:
    import statsmodels.api as sm
except ImportError:
    import subprocess; subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'statsmodels'])
    import statsmodels.api as sm

df = con.execute("""
    SELECT CAST(proveedor_unico AS INTEGER) AS y,
           anio_proceso, tipo_contratacion_grp, entidad_ruc,
           desviacion_precio_log_z_sinfuga,
           dominancia_maxima_historica, win_rate, concentracion_rival_habitual, mad_benford,
           CAST(proveedor_es_nuevo_en_mercado AS INTEGER)          AS nuevo_en_mercado,
           proveedor_antiguedad_dias,
           CAST(proveedor_provincia_distinta_entidad AS INTEGER)   AS provincia_distinta,
           ratio_monto_capital,
           log_monto
    FROM bm3
    WHERE tiene_periodo_competitivo AND proveedor_unico IS NOT NULL AND anio_proceso >= 2016
""").df()
print(f'Procesos competitivos 2016+: {len(df):,} | % proveedor único: {100*df.y.mean():.1f}%')

# Variables continuas -> estandarizadas; los nulos se tratan con indicador explícito (+ valor 0)
cont = {
    'desviacion_precio':  df['desviacion_precio_log_z_sinfuga'].abs(),
    'dominancia':         df['dominancia_maxima_historica'],
    'win_rate':           df['win_rate'],
    'competidor_habitual':df['concentracion_rival_habitual'],
    'benford_mad':        df['mad_benford'],
    'antiguedad_log':     np.log1p(df['proveedor_antiguedad_dias'].clip(lower=0)),
    'ratio_monto_capital_log': np.log1p(df['ratio_monto_capital'].clip(lower=0)),
}
X = pd.DataFrame(index=df.index)
for k, s in cont.items():
    s = s.astype(float)
    X[f'{k}_falta'] = s.isna().astype(int)
    z = (s - s.mean()) / s.std()
    X[k] = z.fillna(0)
X['nuevo_en_mercado'] = df['nuevo_en_mercado'].fillna(0)
X['provincia_distinta'] = df['provincia_distinta'].fillna(0)
X['log_monto'] = ((df['log_monto'] - df['log_monto'].mean()) / df['log_monto'].std()).fillna(0)
# Efectos fijos de año y tipo de contratación (controlan el cambio de publicación por año)
X = pd.concat([X, pd.get_dummies(df['anio_proceso'], prefix='anio', drop_first=True, dtype=int),
                  pd.get_dummies(df['tipo_contratacion_grp'], prefix='tipo', drop_first=True, dtype=int)], axis=1)
X = X.loc[:, X.std() > 0]
X = sm.add_constant(X)

grupos = pd.factorize(df['entidad_ruc'])[0]     # errores agrupados por entidad compradora
res = sm.Logit(df['y'], X).fit(disp=0, cov_type='cluster', cov_kwds={'groups': grupos})

claves = list(cont.keys()) + ['nuevo_en_mercado', 'provincia_distinta', 'log_monto']
claves += [f'{k}_falta' for k in cont if f'{k}_falta' in X.columns]   # también se reporta el efecto de "no tener historial"
claves = [c for c in claves if c in X.columns]
tabla = pd.DataFrame({
    'odds_ratio': np.exp(res.params[claves]),
    'IC95_inf':   np.exp(res.conf_int().loc[claves, 0]),
    'IC95_sup':   np.exp(res.conf_int().loc[claves, 1]),
    'p_valor':    res.pvalues[claves],
}).round(4)
tabla['lectura'] = np.where(tabla.p_valor >= 0.05, 'no significativa',
                   np.where(tabla.odds_ratio > 1, 'MÁS riesgo (valida)', 'menos riesgo (invertida)'))
print(f'\nPseudo R2: {res.prsquared:.4f}\n')
print(tabla.to_string())

In [ ]:
# ============ Celda 3 — Diagnóstico de separación + validación Fazekas corregida ============
import numpy as np, pandas as pd
import statsmodels.api as sm

base = con.execute("""
    SELECT CAST(proveedor_unico AS INTEGER) AS y, anio_proceso, tipo_contratacion_grp, entidad_ruc,
           desviacion_precio_log_z_sinfuga, dominancia_maxima_historica, win_rate,
           concentracion_rival_habitual, mad_benford,
           CAST(proveedor_es_nuevo_en_mercado AS INTEGER)        AS nuevo_en_mercado,
           proveedor_antiguedad_dias,
           CAST(proveedor_provincia_distinta_entidad AS INTEGER) AS provincia_distinta,
           ratio_monto_capital, log_monto
    FROM bm3
    WHERE tiene_periodo_competitivo AND proveedor_unico IS NOT NULL AND anio_proceso >= 2016
""").df()

# --- 1) ¿Dónde está la separación? (grupos donde y es siempre 0 o siempre 1) ---
print('% proveedor único según faltantes y tipo de contratación:')
diag = {
    'desviacion_precio falta': base['desviacion_precio_log_z_sinfuga'].isna(),
    'dominancia falta':        base['dominancia_maxima_historica'].isna(),
    'competidor falta':        base['concentracion_rival_habitual'].isna(),
    'win_rate falta':          base['win_rate'].isna(),
    'benford falta':           base['mad_benford'].isna(),
    'antiguedad falta':        base['proveedor_antiguedad_dias'].isna(),
    'capital falta':           base['ratio_monto_capital'].isna(),
}
for k, m in diag.items():
    print(f'  {k:26s} filas={m.sum():>8,}  %unico si falta={100*base.y[m].mean():5.1f}  %unico si hay dato={100*base.y[~m].mean():5.1f}')
print('  dominancia y competidor faltan en las mismas filas:',
      (diag['dominancia falta'] == diag['competidor falta']).all())
t = base.groupby('tipo_contratacion_grp')['y'].agg(['size', 'mean']).round(3)
print('\nPor tipo de contratación:\n', t.to_string())

# --- 2) Muestra limpia: con desviación de precio y sin tipos que separan perfecto ---
tipos_ok = t[(t['mean'] > 0) & (t['mean'] < 1)].index
d = base[base['desviacion_precio_log_z_sinfuga'].notna() & base['tipo_contratacion_grp'].isin(tipos_ok)].copy()
print(f'\nMuestra de estimación: {len(d):,} de {len(base):,} procesos')

def z(s):
    s = s.astype(float); return ((s - s.mean()) / s.std()).fillna(0)

def armar_X(d, relacionales):
    X = pd.DataFrame(index=d.index)
    X['desviacion_precio']      = z(d['desviacion_precio_log_z_sinfuga'].abs())
    X['benford_mad']            = z(d['mad_benford'])
    X['benford_falta']          = d['mad_benford'].isna().astype(int)
    X['antiguedad_log']         = z(np.log1p(d['proveedor_antiguedad_dias'].clip(lower=0)))
    X['antiguedad_falta']       = d['proveedor_antiguedad_dias'].isna().astype(int)
    X['ratio_monto_capital_log']= z(np.log1p(d['ratio_monto_capital'].clip(lower=0)))
    X['no_es_sociedad']         = d['ratio_monto_capital'].isna().astype(int)
    X['nuevo_en_mercado']       = d['nuevo_en_mercado'].fillna(0)
    X['provincia_distinta']     = d['provincia_distinta'].fillna(0)
    X['log_monto']              = z(d['log_monto'])
    if relacionales:
        X['win_rate']            = z(d['win_rate'])
        X['win_rate_falta']      = d['win_rate'].isna().astype(int)
        X['dominancia']          = z(d['dominancia_maxima_historica'])
        X['competidor_habitual'] = z(d['concentracion_rival_habitual'])
        X['sin_pares_recurrentes'] = d['dominancia_maxima_historica'].isna().astype(int)  # una sola bandera (antes estaba duplicada)
    X = pd.concat([X, pd.get_dummies(d['anio_proceso'], prefix='anio', drop_first=True, dtype=int),
                      pd.get_dummies(d['tipo_contratacion_grp'], prefix='tipo', drop_first=True, dtype=int)], axis=1)
    X = X.loc[:, X.std() > 0]
    return sm.add_constant(X)

# Signo esperado según la teoría: +1 = valor alto es riesgo; -1 = valor BAJO es riesgo; 0 = control
signo = {'desviacion_precio': 1, 'benford_mad': 1, 'benford_falta': 0, 'antiguedad_log': -1, 'antiguedad_falta': 0,
         'ratio_monto_capital_log': 1, 'no_es_sociedad': 0, 'nuevo_en_mercado': 1, 'provincia_distinta': 1,
         'log_monto': 0, 'win_rate': 1, 'win_rate_falta': 0, 'dominancia': 1, 'competidor_habitual': 1,
         'sin_pares_recurrentes': 0}

grupos = pd.factorize(d['entidad_ruc'])[0]
resultados = {}
for nombre, rel in [('M1 sin variables de puja', False), ('M2 con variables de puja', True)]:
    X = armar_X(d, rel)
    res = sm.Logit(d['y'], X).fit(method='newton', maxiter=200, disp=0,
                                  cov_type='cluster', cov_kwds={'groups': grupos})
    claves = [c for c in signo if c in X.columns]
    ci = res.conf_int()
    tb = pd.DataFrame({'odds_ratio': np.exp(res.params[claves]),
                       'IC95_inf': np.exp(ci.loc[claves, 0]), 'IC95_sup': np.exp(ci.loc[claves, 1]),
                       'p_valor': res.pvalues[claves]}).round(4)
    def leer(r):
        s = signo[r.name]
        if r.p_valor >= 0.05: return 'no significativa'
        if s == 0: return 'control / faltante'
        return 'VALIDA (signo esperado)' if (r.odds_ratio > 1) == (s > 0) else 'invertida'
    tb['lectura'] = tb.apply(leer, axis=1)
    resultados[nombre] = res
    print(f'\n===== {nombre} | convergió: {res.mle_retvals["converged"]} | Pseudo R2: {res.prsquared:.4f} =====')
    print(tb.to_string())

In [ ]:
# ============ Celda 4 — Excel de RUC de proveedores para la búsqueda manual del equipo ============
rucs = con.execute("""
    SELECT proveedor_ruc                                  AS ruc,
           ANY_VALUE(proveedor_nombre)                    AS nombre,
           COUNT(*)                                       AS n_procesos,
           ROUND(SUM(monto_final), 2)                     AS monto_total,
           MIN(anio_proceso)                              AS primer_anio,
           MAX(anio_proceso)                              AS ultimo_anio,
           BOOL_OR(proveedor_es_sociedad_registrada)      AS es_sociedad,
           ANY_VALUE(proveedor_estado_actual_sri)         AS estado_sri,
           ANY_VALUE(provincia_proveedor)                 AS provincia
    FROM bm3
    WHERE proveedor_ruc IS NOT NULL AND NOT COALESCE(proveedor_es_entidad_publica, FALSE)
    GROUP BY 1
""").df()
rucs = rucs.sort_values('monto_total', ascending=False).reset_index(drop=True)
rucs['prioridad_monto'] = rucs.index + 1
for c in ['es_incumplido', 'es_adjudicatario_fallido', 'fecha_declaratoria', 'entidad_que_declaro', 'observacion', 'revisado_por']:
    rucs[c] = ''
print(f'Proveedores privados únicos: {len(rucs):,}')

# Muestra aleatoria (semilla fija): sirve para estimar la tasa REAL de incumplidos sin sesgo
muestra = rucs.sample(n=min(500, len(rucs)), random_state=42).sort_values('prioridad_monto')
top = rucs.head(500)

ruta_xlsx = f'{R}/ruc_proveedores_para_revision.xlsx'
with pd.ExcelWriter(ruta_xlsx, engine='openpyxl') as w:
    pd.DataFrame({'instrucciones': [
        'Hoja top_500_monto: los 500 proveedores con más dinero adjudicado (revisar primero).',
        'Hoja muestra_aleatoria_500: 500 proveedores al azar (semilla 42). NO reemplazar ni saltar filas: sirve para estimar la tasa real.',
        'Hoja todos: lista completa, ordenada por monto total.',
        'Llenar es_incumplido / es_adjudicatario_fallido con SI o NO; fecha_declaratoria en formato AAAA-MM-DD.',
        'Anotar en revisado_por el nombre de quien revisó, y la fuente en observacion.',
    ]}).to_excel(w, sheet_name='LEEME', index=False)
    top.to_excel(w, sheet_name='top_500_monto', index=False)
    muestra.to_excel(w, sheet_name='muestra_aleatoria_500', index=False)
    rucs.to_excel(w, sheet_name='todos', index=False)
print('Excel guardado en:', ruta_xlsx)

In [ ]:
# ============ Celda 5 — Reconocer la estructura de los archivos nuevos (sin cargarlos completos) ============
import glob, re
DOC = '05_documentos'

# a) Archivos CPC, CIIU y catastro de fantasmas: hojas y primeras filas crudas (para saber dónde está el encabezado)
for patron in ['*cpc*', '*CPC*', '*ciiu*', '*CIIU*', '*antasma*']:
    for f in sorted(set(glob.glob(os.path.join(DOC, patron)))):
        if not f.lower().endswith(('.xlsx', '.xls')):
            continue
        print(f'\n########## {f} ##########')
        xl = pd.ExcelFile(f)
        print('Hojas:', xl.sheet_names)
        for h in xl.sheet_names[:3]:
            crudo = pd.read_excel(f, sheet_name=h, header=None, nrows=8)
            print(f'--- hoja "{h}" (primeras 8 filas crudas, {crudo.shape[1]} columnas) ---')
            print(crudo.to_string(max_colwidth=40))

# b) JSON originales del OCDS: cuántos hay, dónde, y cómo viene la clasificación de ítems
jsons = glob.glob('**/*.json', recursive=True)
print(f'\n########## JSON encontrados: {len(jsons):,} ##########')
carpetas = pd.Series([os.path.dirname(j) for j in jsons]).value_counts()
print(carpetas.head(10).to_string())
if jsons:
    ej = sorted(jsons, key=os.path.getsize)[len(jsons)//2]      # uno de tamaño mediano
    print('\nArchivo de ejemplo:', ej, f'({os.path.getsize(ej)/1e6:.1f} MB)')
    with open(ej, encoding='utf-8', errors='ignore') as fh:
        trozo = fh.read(3_000_000)                                # solo los primeros ~3 MB
    print('Empieza con:', trozo[:300].replace('\n', ' '))
    m = re.search(r'"classification"\s*:\s*\{[^}]*\}', trozo)
    print('Ejemplo de classification:', m.group(0) if m else 'NO encontrado en los primeros 3 MB')
    m2 = re.search(r'"items"\s*:\s*\[.{0,600}', trozo, re.S)
    print('Ejemplo de items:', m2.group(0)[:600] if m2 else 'NO encontrado')

In [ ]:
# ============ Celda 6 — Extraer el CPC de cada proceso desde los JSON crudos (streaming, reanudable) ============
# Lee cada JSON release por release (no carga el archivo de 1.7 GB en memoria).
# Guarda un parquet por archivo en 03_resultados/cpc_partes/ -> si se corta, al volver a correr salta los ya hechos.
import os, glob, time, json
import pandas as pd
try:
    import ijson
except ImportError:
    import subprocess, sys; subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'ijson']); import ijson
print('Motor ijson:', ijson.backend, '(ideal: yajl2_c, es el rápido)')

SALIDA = os.path.join(R, 'cpc_partes'); os.makedirs(SALIDA, exist_ok=True)

def prefijo_de(ruta):
    """El archivo puede ser una LISTA de paquetes [ {releases:[...]}, ... ] o UN paquete {releases:[...]}"""
    with open(ruta, 'rb') as fh:
        while True:
            c = fh.read(1)
            if not c.isspace():
                return 'item.releases.item' if c == b'[' else 'releases.item'

def monto_item(it):
    try:
        v = float(it.get('unit', {}).get('value', {}).get('amount') or 0)
        q = float(it.get('quantity') or 1)
        return v * q
    except Exception:
        return 0.0

def fila_de(rel, anio_archivo):
    items = (rel.get('tender') or {}).get('items') or []
    if not items:                                          # respaldo: ítems de las adjudicaciones
        items = [i for a in (rel.get('awards') or []) for i in (a.get('items') or [])]
    cpcs = []
    for it in items:
        cl = it.get('classification') or {}
        if str(cl.get('scheme', '')).upper() == 'CPC' and cl.get('id'):
            cpcs.append((str(cl['id']), cl.get('description'), monto_item(it)))
    principal = max(cpcs, key=lambda x: x[2]) if cpcs else (None, None, None)
    cpc5 = {c[0][:5] for c in cpcs}
    return {
        'ocid': rel.get('ocid'), 'fecha_release_json': rel.get('date'), 'anio_archivo': anio_archivo,
        'n_items': len(items), 'n_items_con_cpc': len(cpcs),
        'cpc_principal': principal[0], 'cpc_principal_desc': principal[1],
        'cpc5_principal': principal[0][:5] if principal[0] else None,
        'n_cpc5_distintos': len(cpc5),
        'cpc5_todos': '|'.join(sorted(cpc5)) if cpc5 else None,
    }

archivos = sorted(glob.glob(os.path.join('00_json_crudo', '*.json')))
print(f'{len(archivos)} archivos JSON')
for ruta in archivos:
    nombre = os.path.splitext(os.path.basename(ruta))[0]
    destino = os.path.join(SALIDA, f'{nombre}.parquet')
    if os.path.exists(destino):
        print(f'  {nombre}: ya procesado, se salta'); continue
    anio = ''.join(ch for ch in nombre if ch.isdigit())[:4] or None
    t0, filas = time.time(), []
    with open(ruta, 'rb') as fh:
        for i, rel in enumerate(ijson.items(fh, prefijo_de(ruta), use_float=True)):
            filas.append(fila_de(rel, anio))
            if i and i % 200_000 == 0:
                print(f'    {nombre}: {i:,} releases ({time.time()-t0:.0f} s)')
    df = pd.DataFrame(filas)
    df.to_parquet(destino + '.tmp', index=False); os.replace(destino + '.tmp', destino)
    print(f'  {nombre}: {len(df):,} releases | con CPC: {100*(df.n_items_con_cpc>0).mean():.1f}% | {time.time()-t0:.0f} s')

# Consolidar: una fila por ocid (si un ocid aparece en varios archivos, se queda el release más reciente)
con.execute(f"""
COPY (
  SELECT * EXCLUDE (rn) FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY ocid ORDER BY fecha_release_json DESC NULLS LAST, anio_archivo DESC) AS rn
    FROM read_parquet('{SALIDA.replace(os.sep, '/')}/*.parquet') WHERE ocid IS NOT NULL
  ) WHERE rn = 1
) TO '{R}/cpc_por_ocid.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
""")
con.execute(f"CREATE OR REPLACE VIEW cpc AS SELECT * FROM read_parquet('{R}/cpc_por_ocid.parquet')")
print(con.execute("""
    SELECT COUNT(*) AS ocids, ROUND(100*AVG(CASE WHEN cpc5_principal IS NOT NULL THEN 1 ELSE 0 END),1) AS pct_con_cpc,
           COUNT(DISTINCT cpc5_principal) AS cpc5_distintos
    FROM cpc""").df().to_string(index=False))
print('Cobertura sobre la base maestra (por año):')
print(con.execute("""
    SELECT b.anio_proceso, COUNT(*) AS filas,
           ROUND(100*AVG(CASE WHEN c.cpc5_principal IS NOT NULL THEN 1 ELSE 0 END),1) AS pct_con_cpc
    FROM bm3 b LEFT JOIN cpc c USING (ocid) GROUP BY 1 ORDER BY 1""").df().to_string(index=False))
print(con.execute("SELECT cpc_principal, cpc5_principal, cpc_principal_desc FROM cpc WHERE cpc_principal IS NOT NULL LIMIT 5").df().to_string(index=False))

In [ ]:
# ============ Celda 7 — Cifras descriptivas para redactar los hallazgos de la tesis ============
# Todas las cifras se guardan también en 03_resultados/estadisticas_hallazgos.csv (trazabilidad: cada número del texto sale de aquí)
import pandas as pd
pd.set_option('display.width', 200)
stats = []
def guardar(bloque, df):
    print(f'\n==================== {bloque} ====================')
    print(df.to_string(index=False))
    d = df.copy(); d.insert(0, 'bloque', bloque); stats.append(d)

# Métodos realmente competitivos = los que NO tienen 100% (ni 0%) de proveedor único por diseño legal
con.execute("""
CREATE OR REPLACE TEMP VIEW comp AS
WITH tipos AS (
    SELECT tipo_contratacion_grp FROM bm3
    WHERE tiene_periodo_competitivo AND proveedor_unico IS NOT NULL
    GROUP BY 1 HAVING AVG(CAST(proveedor_unico AS INT)) > 0 AND AVG(CAST(proveedor_unico AS INT)) < 0.99
)
SELECT b.* FROM bm3 b JOIN tipos USING (tipo_contratacion_grp)
WHERE b.tiene_periodo_competitivo AND b.proveedor_unico IS NOT NULL
""")

# A) Universo: cuánto es realmente competitivo, y proveedor único real
guardar('A. Universo y proveedor unico real', con.execute("""
    SELECT (SELECT COUNT(*) FROM bm3)                                             AS procesos_total,
           (SELECT COUNT(*) FROM bm3 WHERE tiene_periodo_competitivo)             AS con_periodo_competitivo,
           (SELECT COUNT(*) FROM comp)                                            AS en_metodos_realmente_competitivos,
           ROUND(100.0*(SELECT COUNT(*) FROM comp)/(SELECT COUNT(*) FROM bm3),1)  AS pct_del_total,
           ROUND(100*(SELECT AVG(CAST(proveedor_unico AS INT)) FROM bm3 WHERE tiene_periodo_competitivo AND proveedor_unico IS NOT NULL),1) AS pct_unico_ingenuo,
           ROUND(100*(SELECT AVG(CAST(proveedor_unico AS INT)) FROM comp),1)      AS pct_unico_real,
           ROUND((SELECT SUM(monto_final) FROM comp WHERE proveedor_unico)/1e6,1) AS monto_millones_con_un_oferente
""").df())
guardar('A2. Proveedor unico real por metodo', con.execute("""
    SELECT tipo_contratacion_grp AS metodo, COUNT(*) AS procesos,
           ROUND(100*AVG(CAST(proveedor_unico AS INT)),1) AS pct_un_solo_oferente,
           ROUND(AVG(n_oferentes),2) AS prom_oferentes
    FROM comp GROUP BY 1 ORDER BY procesos DESC
""").df())
guardar('A3. Proveedor unico real y oferentes por anio', con.execute("""
    SELECT anio_proceso AS anio, COUNT(*) AS procesos,
           ROUND(100*AVG(CAST(proveedor_unico AS INT)),1) AS pct_un_solo_oferente,
           ROUND(AVG(n_oferentes),2) AS prom_oferentes
    FROM comp GROUP BY 1 ORDER BY 1
""").df())

# B) Antigüedad: ¿quién gana solo? (solo proveedores privados)
guardar('B. Proveedor unico por antiguedad del proveedor', con.execute("""
    WITH t AS (
      SELECT CASE WHEN proveedor_antiguedad_dias < 365      THEN '1) < 1 año'
                  WHEN proveedor_antiguedad_dias < 3*365    THEN '2) 1-3 años'
                  WHEN proveedor_antiguedad_dias < 5*365    THEN '3) 3-5 años'
                  WHEN proveedor_antiguedad_dias < 10*365   THEN '4) 5-10 años'
                  WHEN proveedor_antiguedad_dias < 20*365   THEN '5) 10-20 años'
                  ELSE '6) > 20 años' END AS antiguedad, proveedor_unico, monto_final
      FROM comp WHERE proveedor_antiguedad_dias IS NOT NULL AND NOT COALESCE(proveedor_es_entidad_publica, FALSE)
    )
    SELECT antiguedad, COUNT(*) AS procesos,
           ROUND(100*AVG(CAST(proveedor_unico AS INT)),1) AS pct_gano_solo,
           ROUND(100.0*SUM(CAST(proveedor_unico AS INT))/SUM(SUM(CAST(proveedor_unico AS INT))) OVER (),1) AS pct_de_todas_las_victorias_en_solitario
    FROM t GROUP BY 1 ORDER BY 1
""").df())

# C) Entrada a mercado nuevo
guardar('C. Proveedor unico segun entrada a mercado nuevo', con.execute("""
    SELECT CASE WHEN proveedor_es_nuevo_en_mercado THEN 'primer año en ese mercado' ELSE 'ya tenía historial en el mercado' END AS situacion,
           COUNT(*) AS procesos, ROUND(100*AVG(CAST(proveedor_unico AS INT)),1) AS pct_gano_solo
    FROM comp WHERE proveedor_es_nuevo_en_mercado IS NOT NULL AND NOT COALESCE(proveedor_es_entidad_publica, FALSE)
    GROUP BY 1
""").df())
guardar('C2. Cruce: antiguedad x nuevo en mercado', con.execute("""
    SELECT CASE WHEN proveedor_antiguedad_dias >= 10*365 THEN 'empresa de 10+ años' ELSE 'empresa de < 10 años' END AS antiguedad,
           CASE WHEN proveedor_es_nuevo_en_mercado THEN 'nuevo en el mercado' ELSE 'con historial en el mercado' END AS mercado,
           COUNT(*) AS procesos, ROUND(100*AVG(CAST(proveedor_unico AS INT)),1) AS pct_gano_solo
    FROM comp WHERE proveedor_antiguedad_dias IS NOT NULL AND proveedor_es_nuevo_en_mercado IS NOT NULL
               AND NOT COALESCE(proveedor_es_entidad_publica, FALSE)
    GROUP BY 1, 2 ORDER BY 1, 2
""").df())

# D) Competidor habitual por cuartil
guardar('D. Proveedor unico por cuartil de competidor habitual', con.execute("""
    WITH t AS (SELECT NTILE(4) OVER (ORDER BY concentracion_rival_habitual) AS cuartil, concentracion_rival_habitual, proveedor_unico
               FROM comp WHERE concentracion_rival_habitual IS NOT NULL)
    SELECT cuartil, ROUND(MIN(concentracion_rival_habitual),3) AS desde, ROUND(MAX(concentracion_rival_habitual),3) AS hasta,
           COUNT(*) AS procesos, ROUND(100*AVG(CAST(proveedor_unico AS INT)),1) AS pct_gano_solo
    FROM t GROUP BY 1 ORDER BY 1
""").df())

# E) Monto adjudicado frente al capital de la empresa (toda la base, no solo competitivos)
guardar('E. Monto adjudicado vs capital suscrito', con.execute("""
    SELECT CASE WHEN ratio_monto_capital > 100 THEN '> 100 veces el capital'
                WHEN ratio_monto_capital > 10  THEN '10-100 veces'
                WHEN ratio_monto_capital > 1   THEN '1-10 veces'
                ELSE '<= capital' END AS rango,
           COUNT(*) AS procesos, COUNT(DISTINCT proveedor_ruc) AS empresas,
           ROUND(SUM(monto_final)/1e6,1) AS monto_millones,
           ROUND(quantile_cont(proveedor_capital_suscrito, 0.5),0) AS capital_mediano
    FROM bm3 WHERE ratio_monto_capital IS NOT NULL
    GROUP BY 1 ORDER BY MIN(ratio_monto_capital)
""").df())

# F) Concentración del mercado: cuánto del dinero se llevan los proveedores más grandes
guardar('F. Concentracion del gasto en los proveedores mas grandes', con.execute("""
    WITH p AS (SELECT proveedor_ruc, SUM(monto_final) AS m FROM bm3
               WHERE proveedor_ruc IS NOT NULL AND NOT COALESCE(proveedor_es_entidad_publica, FALSE) GROUP BY 1),
         r AS (SELECT m, PERCENT_RANK() OVER (ORDER BY m DESC) AS pr FROM p)
    SELECT COUNT(*) AS proveedores_privados,
           ROUND(100*SUM(CASE WHEN pr < 0.01 THEN m END)/SUM(m),1) AS pct_monto_top_1pct,
           ROUND(100*SUM(CASE WHEN pr < 0.10 THEN m END)/SUM(m),1) AS pct_monto_top_10pct,
           ROUND(100*SUM(CASE WHEN pr >= 0.50 THEN m END)/SUM(m),1) AS pct_monto_mitad_inferior
    FROM r
""").df())

# G) Contratos entre entidades públicas (proveedor que también es comprador)
guardar('G. Peso de las entidades publicas como proveedoras', con.execute("""
    SELECT ROUND(100*AVG(CASE WHEN proveedor_es_entidad_publica THEN 1 ELSE 0 END),2) AS pct_procesos,
           ROUND(100*SUM(CASE WHEN proveedor_es_entidad_publica THEN monto_final ELSE 0 END)/SUM(monto_final),1) AS pct_monto
    FROM bm3 WHERE proveedor_ruc IS NOT NULL
""").df())

# H) Fantasmas a nivel EMPRESA (para documentar el sesgo de la etiqueta)
guardar('H. Fantasmas por empresa (anonimizadas)', con.execute("""
    WITH f AS (SELECT proveedor_ruc, COUNT(*) AS procesos, ROUND(SUM(monto_final),0) AS monto,
                      MIN(anio_proceso) AS desde, MAX(anio_proceso) AS hasta,
                      MODE(metodo_contratacion) AS metodo_principal, MODE(provincia_proveedor) AS provincia
               FROM bm3 WHERE proveedor_es_fantasma GROUP BY 1)
    SELECT 'F' || LPAD(CAST(ROW_NUMBER() OVER (ORDER BY procesos DESC) AS VARCHAR), 2, '0') AS empresa,
           procesos, ROUND(100.0*procesos/SUM(procesos) OVER (),1) AS pct_de_filas_fantasma,
           monto, desde, hasta, metodo_principal, provincia
    FROM f ORDER BY procesos DESC
""").df())

todo = pd.concat(stats, ignore_index=True)
todo.to_csv(f'{R}/estadisticas_hallazgos.csv', index=False, encoding='utf-8-sig')
print(f'\nGuardado: {R}/estadisticas_hallazgos.csv')

In [ ]:
# ============ Celda 8 — (a) hueco de CPC 2018-2020, (b) fantasmas que siguieron contratando, (c) CPC a la base maestra ============
import glob, os
import pandas as pd

# (a) ¿Por qué falta el CPC en 2018-2020? ¿No hay ítems, o hay ítems sin clasificación CPC?
print(con.execute(f"""
    SELECT anio_archivo, COUNT(*) AS releases,
           ROUND(100*AVG(CASE WHEN n_items = 0 THEN 1 ELSE 0 END),1)                          AS pct_sin_items,
           ROUND(100*AVG(CASE WHEN n_items > 0 AND n_items_con_cpc = 0 THEN 1 ELSE 0 END),1)  AS pct_items_sin_cpc,
           ROUND(100*AVG(CASE WHEN n_items_con_cpc > 0 THEN 1 ELSE 0 END),1)                  AS pct_con_cpc
    FROM read_parquet('{R}/cpc_partes/*.parquet') GROUP BY 1 ORDER BY 1
""").df().to_string(index=False))

# (b) Fantasmas: ¿siguieron ganando contratos DESPUÉS de que el SRI las notificó?
cand = [f for f in glob.glob('**/*antasma*.csv', recursive=True)]
print('\nArchivos de fantasmas encontrados:', cand)
if cand:
    fx = pd.read_csv(cand[0], sep='|', encoding='latin-1', dtype=str)
    col_f = [c for c in fx.columns if 'NOTIFICACION' in c.upper()]
    print('Columnas de fecha de notificación:', col_f, '| ejemplo:', fx[col_f[0]].dropna().head(3).tolist() if col_f else None)
    fx['ruc'] = fx['NUMERO_RUC'].str.replace(r'\.0$', '', regex=True).str.strip().str.zfill(13)
    fx['fecha_notif'] = pd.to_datetime(fx[col_f[0]], dayfirst=True, errors='coerce')
    con.register('fx_notif', fx[['ruc', 'fecha_notif']].dropna().drop_duplicates('ruc'))
    tras = con.execute("""
        WITH f AS (
            SELECT b.proveedor_ruc, n.fecha_notif, TRY_CAST(SUBSTR(CAST(b.fecha_release AS VARCHAR), 1, 19) AS TIMESTAMP) AS fecha, b.monto_final
            FROM bm3 b JOIN fx_notif n ON b.proveedor_ruc = n.ruc WHERE b.proveedor_es_fantasma)
        SELECT 'F' || LPAD(CAST(ROW_NUMBER() OVER (ORDER BY COUNT(*) DESC) AS VARCHAR), 2, '0') AS empresa,
               CAST(MIN(fecha_notif) AS DATE) AS notificada_sri,
               COUNT(*) AS procesos,
               SUM(CASE WHEN fecha > fecha_notif THEN 1 ELSE 0 END) AS procesos_despues_de_notificada,
               ROUND(SUM(CASE WHEN fecha > fecha_notif THEN monto_final ELSE 0 END), 0) AS monto_despues,
               CAST(MAX(fecha) AS DATE) AS ultimo_proceso
        FROM f GROUP BY proveedor_ruc ORDER BY procesos DESC
    """).df()
    print(tras.to_string(index=False))
    print('Total procesos después de la notificación:', int(tras.procesos_despues_de_notificada.sum()),
          '| monto:', f"{tras.monto_despues.sum():,.0f}")
    print('Nota: fecha_release es la última actualización del proceso, no la firma; confirmar caso por caso.')

# (c) Agregar el CPC a base_maestra_v3 (una fila por ocid -> no cambia el número de filas)
cols_cpc = ['cpc_principal', 'cpc5_principal', 'cpc_principal_desc', 'n_cpc5_distintos', 'n_items']
ya_estan = [c for c in cols_cpc if c in [x[0] for x in con.execute("DESCRIBE bm3").fetchall()]]
excluir = f"EXCLUDE ({', '.join(ya_estan)})" if ya_estan else ''      # si se corre dos veces, no duplica columnas
con.execute(f"""
COPY (
  SELECT b.* {excluir}, c.cpc_principal, c.cpc5_principal, c.cpc_principal_desc, c.n_cpc5_distintos, c.n_items
  FROM bm3 b LEFT JOIN read_parquet('{R}/cpc_por_ocid.parquet') c USING (ocid)
) TO '{R}/base_maestra_v3_tmp.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
""")
n_antes = con.execute("SELECT COUNT(*) FROM bm3").fetchone()[0]
con.execute("DROP VIEW IF EXISTS bm3")
os.replace(f'{R}/base_maestra_v3_tmp.parquet', f'{R}/base_maestra_v3.parquet')
con.execute(f"CREATE OR REPLACE VIEW bm3 AS SELECT * FROM read_parquet('{R}/base_maestra_v3.parquet')")
n_desp = con.execute("SELECT COUNT(*) FROM bm3").fetchone()[0]
print(f'\nbase_maestra_v3 con CPC: {n_desp:,} filas (antes {n_antes:,}) | columnas:',
      len(con.execute("DESCRIBE bm3").fetchall()))
assert n_antes == n_desp

In [ ]:
# ============ Celda 9 — Excel para la revisión de Proveedores Incumplidos y Adjudicatarios Fallidos ============
import numpy as np, pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.worksheet.datavalidation import DataValidation

REVISORES = ['Bayron', 'Santiago', 'Andres']            # cambia los nombres si hace falta

rucs = con.execute("""
    SELECT proveedor_ruc AS ruc, ANY_VALUE(proveedor_nombre) AS nombre_sercop,
           COUNT(*) AS n_procesos, ROUND(SUM(monto_final), 2) AS monto_total,
           MIN(anio_proceso) AS primer_anio, MAX(anio_proceso) AS ultimo_anio,
           ANY_VALUE(provincia_proveedor) AS provincia
    FROM bm3
    WHERE proveedor_ruc IS NOT NULL AND NOT COALESCE(proveedor_es_entidad_publica, FALSE)
    GROUP BY 1
""").df()

# Orden de revisión: primero una muestra aleatoria (sirve para estimar la tasa real aunque no se termine todo),
# luego el resto de mayor a menor monto. Se reparte en ronda para que cada hoja tenga la misma mezcla.
rng = np.random.default_rng(42)
rucs['muestra_aleatoria'] = 'NO'
rucs.loc[rng.choice(rucs.index, size=min(600, len(rucs)), replace=False), 'muestra_aleatoria'] = 'SI'
rucs = pd.concat([rucs[rucs.muestra_aleatoria == 'SI'].sample(frac=1, random_state=42),
                  rucs[rucs.muestra_aleatoria == 'NO'].sort_values('monto_total', ascending=False)], ignore_index=True)
rucs['orden_revision'] = rucs.index + 1
rucs['revisor'] = [REVISORES[i % 3] for i in range(len(rucs))]

# Columnas a llenar = lo que muestra la consulta del portal SERCOP
llenar = ['encontrado', 'n_registros', 'estado', 'reportado_por', 'resolucion', 'codigo_proceso',
          'fecha_desde', 'fecha_hasta', 'motivo_resumen', 'en_contraloria', 'revisado_por', 'fecha_revision', 'observacion']
for c in llenar:
    rucs[c] = None
orden_cols = ['orden_revision', 'muestra_aleatoria', 'ruc', 'nombre_sercop', 'n_procesos', 'monto_total',
              'primer_anio', 'ultimo_anio', 'provincia'] + llenar

ruta = f'{R}/revision_incumplidos_SERCOP.xlsx'
instr = pd.DataFrame({'INSTRUCCIONES': [
    '1. Cada revisor trabaja SOLO su hoja, en el orden de la columna orden_revision (de arriba hacia abajo).',
    '2. Las primeras filas marcadas muestra_aleatoria = SI son las más importantes: no las saltes. Permiten estimar la tasa real aunque no se revise todo.',
    '3. Buscar cada RUC en el portal del SERCOP: Proveedores Incumplidos y Adjudicatarios Fallidos -> Por RUC.',
    '4. encontrado: SI si aparece en la tabla "reportados al INCOP/SERCOP"; NO si no aparece. Si no aparece, dejar el resto vacío.',
    '5. n_registros: cuántas filas aparecen para ese RUC. Si hay más de una, llenar aquí la PRIMERA y copiar las demás en la hoja registros_adicionales.',
    '6. estado: copiar la columna Estado del portal (Incumplido / Adjudicatario Fallido).',
    '7. reportado_por: entidad de la columna "Reportado por". resolucion: número de resolución tal cual.',
    '8. codigo_proceso: el código que aparece dentro del Motivo (ej. COTBS-CELEGU-240-19). Es clave: permite unirlo con el proceso en la base.',
    '9. fecha_desde y fecha_hasta: formato AAAA-MM-DD (ej. 2021-10-15).',
    '10. motivo_resumen: una frase corta (no hace falta copiar todo el texto).',
    '11. en_contraloria: SI / NO según la segunda tabla (reportados a Contraloría).',
    '12. revisado_por y fecha_revision: tu nombre y la fecha en que revisaste.',
]})
extra = pd.DataFrame(columns=['ruc', 'estado', 'reportado_por', 'resolucion', 'codigo_proceso',
                              'fecha_desde', 'fecha_hasta', 'motivo_resumen', 'revisado_por'])
with pd.ExcelWriter(ruta, engine='openpyxl') as w:
    instr.to_excel(w, sheet_name='INSTRUCCIONES', index=False)
    for r_ in REVISORES:
        rucs[rucs.revisor == r_][orden_cols].to_excel(w, sheet_name=r_, index=False)
    extra.to_excel(w, sheet_name='registros_adicionales', index=False)

# Formato: encabezados, filtros, paneles fijos, listas desplegables y hoja de avance
wb = load_workbook(ruta)
azul, amarillo = PatternFill('solid', fgColor='1F4E79'), PatternFill('solid', fgColor='FFF2CC')
for ws in wb.worksheets:
    for c in ws[1]:
        c.font = Font(bold=True, color='FFFFFF'); c.fill = azul
        c.alignment = Alignment(wrap_text=True, vertical='center')
    ws.freeze_panes = 'D2'
    ws.auto_filter.ref = ws.dimensions
wb['INSTRUCCIONES'].column_dimensions['A'].width = 140
anchos = {'A': 9, 'B': 10, 'C': 15, 'D': 38, 'E': 9, 'F': 14, 'G': 8, 'H': 8, 'I': 14}
for r_ in REVISORES:
    ws = wb[r_]; n = ws.max_row
    for col, a in anchos.items(): ws.column_dimensions[col].width = a
    for col in 'JKLMNOPQRSTUV': ws.column_dimensions[col].width = 16
    ws.column_dimensions['R'].width = 40
    for fila in ws.iter_rows(min_row=2, max_row=n, min_col=10, max_col=22):
        for c in fila: c.fill = amarillo                 # amarillo = columnas para llenar
    for fila in ws.iter_rows(min_row=2, max_row=n, min_col=6, max_col=6):
        for c in fila: c.number_format = '#,##0.00'
    listas = {'J': '"SI,NO"', 'L': '"Incumplido,Adjudicatario Fallido"', 'S': '"SI,NO"'}
    for col, formula in listas.items():
        dv = DataValidation(type='list', formula1=formula, allow_blank=True); ws.add_data_validation(dv)
        dv.add(f'{col}2:{col}{n}')

av = wb.create_sheet('AVANCE', 1)
av.append(['Revisor', 'RUC asignados', 'Revisados', '% avance', 'Encontrados (SI)', 'Muestra aleatoria revisada', 'Muestra: encontrados'])
for r_ in REVISORES:
    n = wb[r_].max_row
    av.append([r_, f"=COUNTA('{r_}'!C2:C{n})", f"=COUNTA('{r_}'!J2:J{n})", f"=IFERROR(C{av.max_row+1}/B{av.max_row+1},0)",
               f"=COUNTIF('{r_}'!J2:J{n},\"SI\")", f"=COUNTIFS('{r_}'!B2:B{n},\"SI\",'{r_}'!J2:J{n},\"<>\")",
               f"=COUNTIFS('{r_}'!B2:B{n},\"SI\",'{r_}'!J2:J{n},\"SI\")"])
av.append(['TOTAL', '=SUM(B2:B4)', '=SUM(C2:C4)', '=IFERROR(C5/B5,0)', '=SUM(E2:E4)', '=SUM(F2:F4)', '=SUM(G2:G4)'])
av.append([]); av.append(['Tasa estimada de incumplidos (muestra aleatoria)', '=IFERROR(G5/F5,0)'])
for c in av[1]: c.font = Font(bold=True, color='FFFFFF'); c.fill = azul
for fila in range(2, 6): av[f'D{fila}'].number_format = '0.0%'
av['B7'].number_format = '0.0%'; av['A7'].font = Font(bold=True)
for col, a in zip('ABCDEFG', [44, 14, 12, 10, 16, 24, 20]): av.column_dimensions[col].width = a
wb.save(ruta)

print(f'Excel guardado: {ruta}')
print(rucs.groupby('revisor').agg(ruc=('ruc', 'size'), muestra=('muestra_aleatoria', lambda s: (s == 'SI').sum()),
                                  monto_millones=('monto_total', lambda s: round(s.sum() / 1e6, 1))).to_string())

In [ ]:
# ============ Celda 8b — Fantasmas: ¿contratos después de la notificación del SRI? (fechas vienen como número de Excel) ============
fx = pd.read_csv('05_documentos/SRI_Catastro_Empresas_Fantasmas.csv', sep='|', encoding='latin-1', dtype=str)
fx['ruc'] = fx['NUMERO_RUC'].str.replace(r'\.0$', '', regex=True).str.strip().str.zfill(13)
def fecha_excel(s):                                   # 43003 -> 2017-09-25 (días desde 1899-12-30)
    return pd.to_datetime(pd.to_numeric(s, errors='coerce'), unit='D', origin='1899-12-30')
fx['f_oficio'] = fecha_excel(fx['FECHA_NOTIFICACION_OFICIO'])          # primer aviso del SRI
fx['f_resolucion'] = fecha_excel(fx['FECHA_NOTIFICACION_RESOLUCION'])  # declaración formal
print(fx[['FECHA_NOTIFICACION_OFICIO', 'f_oficio', 'FECHA_NOTIFICACION_RESOLUCION', 'f_resolucion']].dropna().head(3).to_string(index=False))
con.register('fx_notif', fx[['ruc', 'f_oficio', 'f_resolucion']].drop_duplicates('ruc'))
tras = con.execute("""
    WITH f AS (
        SELECT b.proveedor_ruc, n.f_oficio, n.f_resolucion, b.monto_final,
               TRY_CAST(SUBSTR(CAST(b.fecha_release AS VARCHAR), 1, 19) AS TIMESTAMP) AS fecha
        FROM bm3 b JOIN fx_notif n ON b.proveedor_ruc = n.ruc WHERE b.proveedor_es_fantasma)
    SELECT 'F' || LPAD(CAST(ROW_NUMBER() OVER (ORDER BY COUNT(*) DESC) AS VARCHAR), 2, '0') AS empresa,
           COUNT(*) AS procesos,
           CAST(MIN(fecha) AS DATE) AS primer_proceso, CAST(MAX(fecha) AS DATE) AS ultimo_proceso,
           CAST(MIN(f_oficio) AS DATE) AS oficio_sri, CAST(MIN(f_resolucion) AS DATE) AS resolucion_sri,
           SUM(CASE WHEN fecha > f_oficio THEN 1 ELSE 0 END)     AS procesos_tras_oficio,
           SUM(CASE WHEN fecha > f_resolucion THEN 1 ELSE 0 END) AS procesos_tras_resolucion,
           ROUND(SUM(CASE WHEN fecha > f_resolucion THEN monto_final ELSE 0 END), 0) AS monto_tras_resolucion
    FROM f GROUP BY proveedor_ruc ORDER BY procesos DESC
""").df()
print(tras.to_string(index=False))
print('\nTotal procesos tras el oficio:', int(tras.procesos_tras_oficio.sum()),
      '| tras la resolución:', int(tras.procesos_tras_resolucion.sum()),
      '| monto tras la resolución:', f"{tras.monto_tras_resolucion.sum():,.0f}")

In [ ]:
# ============ Celda 8c — Verificar los contratos de F04 posteriores a la resolución del SRI ============
fx['f_reactivacion'] = fecha_excel(fx['FECHA_NOTIFICACION_REACTIVACION'])
f04 = tras.loc[tras.empresa == 'F04']
# Recuperar el RUC de F04 (mismo orden que la Celda 8b: el 4º con más procesos)
ruc_f04 = con.execute("""
    SELECT proveedor_ruc FROM bm3 WHERE proveedor_es_fantasma
    GROUP BY 1 ORDER BY COUNT(*) DESC LIMIT 1 OFFSET 3""").fetchone()[0]
info = fx.loc[fx.ruc == ruc_f04, ['f_oficio', 'f_resolucion', 'f_reactivacion']]
print('Fechas SRI de F04:\n', info.to_string(index=False))
print('Estado actual en el catastro SRI:',
      con.execute("SELECT ANY_VALUE(proveedor_estado_actual_sri) FROM bm3 WHERE proveedor_ruc = ?", [ruc_f04]).fetchone()[0])

detalle = con.execute("""
    SELECT tender_id, entidad_nombre, metodo_contratacion, tipo_contratacion_grp,
           CAST(fecha_inicio_tender AS DATE)                                          AS inicio_tender,
           CAST(TRY_CAST(SUBSTR(CAST(fecha_release AS VARCHAR),1,19) AS TIMESTAMP) AS DATE) AS fecha_release,
           ROUND(monto_final, 2) AS monto, tender_status, etapas
    FROM bm3 WHERE proveedor_ruc = ?
    ORDER BY fecha_release
""", [ruc_f04]).df()
res = info['f_resolucion'].iloc[0]
detalle['despues_de_resolucion'] = pd.to_datetime(detalle['fecha_release']) > res
print(f'\nTodos los procesos de F04 (resolución SRI: {res.date()}):')
print(detalle.to_string(index=False))
detalle.to_csv(f'{R}/verificacion_F04.csv', index=False, encoding='utf-8-sig')
print(f'\nGuardado para revisión manual: {R}/verificacion_F04.csv')

In [ ]:
# ============ Celda 10 — Índice de Riesgo Compuesto (CRI) en tres dimensiones ============
# Cada bandera se convierte en un valor entre 0 y 1 (percentil DENTRO de su año, o 0/1 si es sí/no).
# Cada dimensión = promedio de las banderas DISPONIBLES en ese proceso (un nulo no cuenta como 0 ni como 1).
# CRI = promedio de las dimensiones disponibles. Pesos iguales: simple, transparente y defendible (Fazekas).
import glob, os
import pandas as pd

# Proveedores que comparten representante legal con un competidor (Superintendencia, notebook 4)
p = pd.read_parquet(f'{R}/pares_mismo_representante.parquet')
cols_ruc = [c for c in p.columns if 'ruc' in c.lower()]
print('Columnas RUC en pares_mismo_representante:', cols_ruc)
mismo_rep = pd.DataFrame({'ruc': pd.unique(p[cols_ruc].values.ravel())}).dropna()
con.register('mismo_rep', mismo_rep)
print('Proveedores con representante compartido:', len(mismo_rep))

con.execute(f"""
COPY (
WITH base AS (
  SELECT ocid, proveedor_ruc, entidad_ruc, anio_proceso, monto_final, tipo_contratacion_grp,
         proveedor_es_fantasma,
         -- ---------- D1 COMPETENCIA (validadas contra proveedor único) ----------
         CASE WHEN proveedor_es_nuevo_en_mercado IS NULL THEN NULL
              WHEN proveedor_es_nuevo_en_mercado THEN 1.0 ELSE 0.0 END                         AS f_nuevo_mercado,
         CASE WHEN concentracion_rival_habitual IS NULL THEN NULL
              ELSE PERCENT_RANK() OVER (PARTITION BY anio_proceso, concentracion_rival_habitual IS NULL
                                        ORDER BY concentracion_rival_habitual) END            AS f_rival_habitual,
         CASE WHEN win_rate IS NULL OR n_participaciones < 5 THEN NULL          -- mínimo 5 participaciones: evita "raro por azar"
              ELSE PERCENT_RANK() OVER (PARTITION BY anio_proceso, (win_rate IS NULL OR n_participaciones < 5)
                                        ORDER BY win_rate) END                                 AS f_win_rate,
         CASE WHEN NOT tiene_periodo_competitivo OR proveedor_unico IS NULL THEN NULL
              WHEN proveedor_unico THEN 1.0 ELSE 0.0 END                                       AS f_un_solo_oferente,
         -- ---------- D2 CAPACIDAD DEL PROVEEDOR ----------
         CASE WHEN ratio_monto_capital IS NULL THEN NULL
              WHEN ratio_monto_capital > 100 THEN 1.0 ELSE 0.0 END                             AS f_monto_100x_capital,
         CASE WHEN proveedor_antiguedad_dias IS NULL OR proveedor_es_entidad_publica THEN NULL
              WHEN proveedor_antiguedad_dias < 365 THEN 1.0 ELSE 0.0 END                       AS f_empresa_menor_1_anio,
         CASE WHEN proveedor_contrato_tras_suspension IS NULL THEN NULL
              WHEN proveedor_contrato_tras_suspension THEN 1.0 ELSE 0.0 END                    AS f_contrato_tras_suspension,
         CASE WHEN NOT COALESCE(proveedor_es_sociedad_registrada, FALSE) THEN NULL
              WHEN proveedor_ruc IN (SELECT ruc FROM mismo_rep) THEN 1.0 ELSE 0.0 END          AS f_representante_compartido,
         -- ---------- D3 PRECIO Y MONTOS ----------
         CASE WHEN desviacion_precio_log_z_sinfuga IS NULL THEN NULL
              ELSE PERCENT_RANK() OVER (PARTITION BY anio_proceso, desviacion_precio_log_z_sinfuga IS NULL
                                        ORDER BY ABS(desviacion_precio_log_z_sinfuga)) END     AS f_desviacion_precio,
         CASE WHEN mad_benford IS NULL OR n_contratos_previos < 10 THEN NULL
              WHEN mad_benford > 0.015 THEN 1.0 ELSE 0.0 END                                   AS f_benford_no_conforme,  -- umbral de Nigrini
         CASE WHEN n_contratos_30d_no_catalogo IS NULL THEN NULL
              WHEN n_contratos_30d_no_catalogo >= 3 THEN 1.0 ELSE 0.0 END                      AS f_fragmentacion
  FROM bm3
  WHERE anio_proceso >= 2016 AND proveedor_ruc IS NOT NULL
),
dims AS (
  SELECT *,
    (COALESCE(f_nuevo_mercado,0)+COALESCE(f_rival_habitual,0)+COALESCE(f_win_rate,0)+COALESCE(f_un_solo_oferente,0))
      / NULLIF((f_nuevo_mercado IS NOT NULL)::INT+(f_rival_habitual IS NOT NULL)::INT+(f_win_rate IS NOT NULL)::INT
               +(f_un_solo_oferente IS NOT NULL)::INT, 0)                                                    AS d1_competencia,
    (COALESCE(f_monto_100x_capital,0)+COALESCE(f_empresa_menor_1_anio,0)+COALESCE(f_contrato_tras_suspension,0)+COALESCE(f_representante_compartido,0))
      / NULLIF((f_monto_100x_capital IS NOT NULL)::INT+(f_empresa_menor_1_anio IS NOT NULL)::INT
               +(f_contrato_tras_suspension IS NOT NULL)::INT+(f_representante_compartido IS NOT NULL)::INT, 0) AS d2_capacidad,
    (COALESCE(f_desviacion_precio,0)+COALESCE(f_benford_no_conforme,0)+COALESCE(f_fragmentacion,0))
      / NULLIF((f_desviacion_precio IS NOT NULL)::INT+(f_benford_no_conforme IS NOT NULL)::INT
               +(f_fragmentacion IS NOT NULL)::INT, 0)                                                      AS d3_precio
  FROM base
)
SELECT *,
  (COALESCE(d1_competencia,0)+COALESCE(d2_capacidad,0)+COALESCE(d3_precio,0))
    / NULLIF((d1_competencia IS NOT NULL)::INT+(d2_capacidad IS NOT NULL)::INT+(d3_precio IS NOT NULL)::INT, 0) AS cri,
  (d1_competencia IS NOT NULL)::INT+(d2_capacidad IS NOT NULL)::INT+(d3_precio IS NOT NULL)::INT              AS n_dimensiones
FROM dims
) TO '{R}/cri_procesos.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
""")
con.execute(f"CREATE OR REPLACE VIEW cri AS SELECT * FROM read_parquet('{R}/cri_procesos.parquet')")

# ---- 1) Cobertura y distribución ----
print('\nCobertura (% de procesos con dato) y promedio de cada bandera y dimensión:')
cols = [c[0] for c in con.execute("DESCRIBE cri").fetchall() if c[0].startswith(('f_', 'd1', 'd2', 'd3', 'cri'))]
print(con.execute('SELECT ' + ', '.join(
    [f"ROUND(100*AVG(CASE WHEN {c} IS NOT NULL THEN 1 ELSE 0 END),1) AS \"{c}|cob\", ROUND(AVG({c}),3) AS \"{c}|prom\"" for c in cols])
    + ' FROM cri').df().T.rename(columns={0: 'valor'}).to_string())
print('\nProcesos por número de dimensiones disponibles:')
print(con.execute("SELECT n_dimensiones, COUNT(*) AS procesos FROM cri GROUP BY 1 ORDER BY 1").df().to_string(index=False))
print('\nCorrelación entre dimensiones (si es baja, miden cosas distintas):')
print(con.execute("""SELECT ROUND(CORR(d1_competencia, d2_capacidad),3) AS d1_d2, ROUND(CORR(d1_competencia, d3_precio),3) AS d1_d3,
                            ROUND(CORR(d2_capacidad, d3_precio),3) AS d2_d3 FROM cri""").df().to_string(index=False))

# ---- 2) Validación externa con fantasmas, a nivel EMPRESA (no por fila) ----
prov = con.execute("""
    SELECT proveedor_ruc, BOOL_OR(proveedor_es_fantasma) AS fantasma, COUNT(*) AS procesos,
           AVG(cri) AS cri_prom, AVG(d1_competencia) AS d1, AVG(d2_capacidad) AS d2, AVG(d3_precio) AS d3
    FROM cri WHERE cri IS NOT NULL GROUP BY 1
""").df()
print(f'\nProveedores con CRI: {len(prov):,} | fantasmas entre ellos: {int(prov.fantasma.sum())}')
for col in ['cri_prom', 'd1', 'd2', 'd3']:
    pr = prov[col].rank(pct=True, ascending=False)      # 0.01 = 1% más riesgoso
    f = pr[prov.fantasma]
    print(f'  {col:9s} percentil mediano fantasmas: {100*f.median():5.1f}% | en top 10%: {int((f <= .10).sum())}/{len(f)} '
          f'(azar: {0.10*len(f):.1f}) | en top 25%: {int((f <= .25).sum())}/{len(f)} (azar: {0.25*len(f):.1f})')
prov.to_parquet(f'{R}/cri_proveedores.parquet', index=False)
print(f'\nGuardado: {R}/cri_procesos.parquet y {R}/cri_proveedores.parquet')

In [ ]:
# ============ Celda 11 — ¿Por qué el CRI no ubica a las fantasmas? + reactivaciones del SRI ============
import numpy as np, pandas as pd

# (a) ¿Cuántas fantasmas del catastro fueron reactivadas por el SRI?
fx['f_reactivacion'] = fecha_excel(fx['FECHA_NOTIFICACION_REACTIVACION'])
nuestras = set(con.execute("SELECT DISTINCT proveedor_ruc FROM bm3 WHERE proveedor_es_fantasma").df().proveedor_ruc)
u = fx.drop_duplicates('ruc')
print(f'Catastro SRI: {len(u):,} RUC | reactivados: {u.f_reactivacion.notna().sum():,} ({100*u.f_reactivacion.notna().mean():.1f}%)')
print(f'Nuestras {len(nuestras)} fantasmas proveedoras | reactivadas: {u[u.ruc.isin(nuestras)].f_reactivacion.notna().sum()}')

# (b) Perfil de banderas: fantasmas vs resto, por EMPRESA (promedio de cada bandera en sus procesos)
flags = [c[0] for c in con.execute("DESCRIBE cri").fetchall() if c[0].startswith(('f_', 'd1', 'd2', 'd3')) or c[0] == 'cri']
perfil = con.execute(f"""
    WITH p AS (SELECT proveedor_ruc, BOOL_OR(proveedor_es_fantasma) AS fantasma,
                      {', '.join(f'AVG({c}) AS {c}' for c in flags)}
               FROM cri GROUP BY 1)
    SELECT CASE WHEN fantasma THEN 'fantasmas' ELSE 'resto' END AS grupo, COUNT(*) AS empresas,
           {', '.join(f'ROUND(AVG({c}),3) AS {c}' for c in flags)}
    FROM p GROUP BY 1
""").df().set_index('grupo').T
perfil['fantasma / resto'] = (perfil['fantasmas'] / perfil['resto']).round(2)
print('\nPromedio de cada bandera por empresa (fantasmas vs resto):')
print(perfil.to_string())

# (c) ¿Cambia si la empresa se resume por su proceso MÁS riesgoso (máximo / percentil 90) en vez del promedio?
agg = con.execute("""
    SELECT proveedor_ruc, BOOL_OR(proveedor_es_fantasma) AS fantasma, COUNT(*) AS n,
           AVG(cri) AS cri_prom, MAX(cri) AS cri_max, quantile_cont(cri, 0.9) AS cri_p90,
           MAX(d1_competencia) AS d1_max, MAX(d2_capacidad) AS d2_max, MAX(d3_precio) AS d3_max
    FROM cri WHERE cri IS NOT NULL GROUP BY 1
""").df()
print(f'\nPercentil de las fantasmas en el ranking de {len(agg):,} proveedores (1% = más riesgoso):')
for col in ['cri_prom', 'cri_p90', 'cri_max', 'd1_max', 'd2_max', 'd3_max']:
    pr = agg[col].rank(pct=True, ascending=False, method='average')
    f = pr[agg.fantasma]
    print(f'  {col:9s} mediana: {100*f.median():5.1f}% | top 10%: {int((f <= .10).sum())}/{len(f)} | top 25%: {int((f <= .25).sum())}/{len(f)}')
print('Nota: con máximo, las empresas con MÁS procesos tienen ventaja (más oportunidades de un valor alto).')

In [ ]:
# ============ Celda 12 — ¿La señal de Benford depende del tamaño de muestra? (y umbral corregido por n) ============
import numpy as np, pandas as pd
P_BENFORD = np.log10(1 + 1 / np.arange(1, 10))            # probabilidad teórica de los dígitos 1..9

fant = set(con.execute("SELECT DISTINCT proveedor_ruc FROM bm3 WHERE proveedor_es_fantasma").df().proveedor_ruc)
bf = con.execute("SELECT proveedor_ruc, anio_aplicable, mad_benford, n_contratos_previos AS n FROM benford").df()
bf['fantasma'] = bf.proveedor_ruc.isin(fant)

# (a) Si el MAD baja cuando n sube, el umbral fijo de Nigrini (0.015) no sirve para comparar proveedores chicos y grandes
bf['tramo_n'] = pd.cut(bf.n, [9, 20, 50, 100, 500, 1000, 5000, np.inf],
                       labels=['10-20', '21-50', '51-100', '101-500', '501-1000', '1001-5000', '>5000'])
print('MAD observado según número de contratos (todos los proveedores-año):')
print(bf.groupby('tramo_n', observed=True).agg(proveedor_anio=('n', 'size'), mad_mediano=('mad_benford', 'median'),
      pct_sobre_0_015=('mad_benford', lambda s: round(100 * (s > 0.015).mean(), 1))).to_string())

# (b) Umbral correcto: percentil 95 del MAD que tendría un proveedor HONESTO (montos que sí siguen Benford) con ese mismo n
rng = np.random.default_rng(42)
def mad_simulado(n, reps=2000):
    props = rng.multinomial(int(n), P_BENFORD, size=reps) / n
    return np.abs(props - P_BENFORD).mean(axis=1)
grid = np.unique(np.round(np.geomspace(10, max(10, bf.n.max()), 120)).astype(int))
p95 = np.array([np.quantile(mad_simulado(n), 0.95) for n in grid])
bf['mad_p95_honesto'] = np.interp(np.log(bf.n), np.log(grid), p95)
bf['mad_exceso'] = bf.mad_benford / bf.mad_p95_honesto     # > 1 = más desviado que el 95% de proveedores honestos de su tamaño
bf['benford_anomalo'] = bf.mad_exceso > 1
print('\nUmbral del 95% para un proveedor honesto, según n:',
      {n: round(float(np.interp(np.log(n), np.log(grid), p95)), 4) for n in (10, 20, 50, 100, 500, 1000, 5000) if n <= grid.max()})
print(f'Proveedores-año marcados con el umbral fijo 0.015: {100*(bf.mad_benford > 0.015).mean():.1f}% '
      f'| con el umbral corregido por n: {100*bf.benford_anomalo.mean():.1f}% (esperado por azar: 5%)')

# (c) Fantasmas vs resto, comparando dentro del mismo tramo de n
print('\nFantasmas vs resto (proveedor-año):')
print(bf.groupby('fantasma').agg(filas=('n', 'size'), n_mediano=('n', 'median'), mad_mediano=('mad_benford', 'median'),
      exceso_mediano=('mad_exceso', 'median'), pct_anomalo=('benford_anomalo', lambda s: round(100 * s.mean(), 1))).to_string())
print('\nMismo cálculo dentro de cada tramo de n:')
print(bf.groupby(['tramo_n', 'fantasma'], observed=True).agg(filas=('n', 'size'),
      pct_anomalo=('benford_anomalo', lambda s: round(100 * s.mean(), 1))).unstack('fantasma').to_string())

bf[['proveedor_ruc', 'anio_aplicable', 'n', 'mad_benford', 'mad_p95_honesto', 'mad_exceso', 'benford_anomalo']] \
  .to_parquet(f'{R}/benford_corregido.parquet', index=False)
print(f'\nGuardado: {R}/benford_corregido.parquet')

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', 'networkx'], check=True)

In [ ]:
# ===== CAPA 2 · PARTE A · Red entidad–proveedor acumulada (sin fuga) =====
import os, time, warnings
import numpy as np, pandas as pd, duckdb
import scipy.sparse as sp
from scipy.sparse.linalg import svds
import networkx as nx
warnings.filterwarnings('ignore')

os.chdir(r'C:\SERCOP_FINAL')
R = '03_resultados'
try:
    con
except NameError:
    con = duckdb.connect(); con.execute("PRAGMA threads=8")
con.execute(f"CREATE OR REPLACE VIEW bm3 AS SELECT * FROM read_parquet('{R}/base_maestra_v3.parquet')")

# 1) Aristas entidad-proveedor por año (1 fila por ocid+proveedor para no duplicar montos)
aristas = con.execute("""
    WITH base AS (
        SELECT ocid, entidad_ruc, proveedor_ruc, anio_proceso,
               MAX(GREATEST(COALESCE(monto_final,0),0)) AS monto
        FROM bm3
         WHERE TRIM(COALESCE(entidad_ruc,'')) <> '' AND TRIM(COALESCE(proveedor_ruc,'')) <> ''
         AND anio_proceso IS NOT NULL
        GROUP BY ocid, entidad_ruc, proveedor_ruc, anio_proceso)
    SELECT entidad_ruc, proveedor_ruc, anio_proceso AS anio,
           COUNT(*) AS n, SUM(monto) AS monto
    FROM base GROUP BY 1,2,3
""").df()
anios = sorted(aristas['anio'].unique())
print(f'Aristas entidad-proveedor-año: {len(aristas):,} | años {anios[0]}–{anios[-1]}')

filas_p, filas_e, resumen = [], [], []
t0 = time.time()
for Y in anios:
    E = aristas[aristas['anio'] < Y]             # SOLO años anteriores
    if E.empty:
        continue
    E = E.groupby(['entidad_ruc','proveedor_ruc'], as_index=False).agg(n=('n','sum'), monto=('monto','sum'))
    ent_ids = pd.Index(E['entidad_ruc'].unique()); pro_ids = pd.Index(E['proveedor_ruc'].unique())
    ie = ent_ids.get_indexer(E['entidad_ruc']); ip = pro_ids.get_indexer(E['proveedor_ruc'])
    nE, nP = len(ent_ids), len(pro_ids)
    w_n = E['n'].values.astype(float); w_m = E['monto'].values.astype(float)

    p_n  = np.bincount(ip, w_n, nP);  p_m = np.bincount(ip, w_m, nP);  p_deg = np.bincount(ip, minlength=nP)
    e_n  = np.bincount(ie, w_n, nE);  e_m = np.bincount(ie, w_m, nE);  e_deg = np.bincount(ie, minlength=nE)
    use_m_p = p_m[ip] > 0; use_m_e = e_m[ie] > 0
    sh_p = np.where(use_m_p, w_m/np.where(p_m[ip]>0,p_m[ip],1), w_n/p_n[ip])   # cuota de la entidad en MI cartera
    sh_e = np.where(use_m_e, w_m/np.where(e_m[ie]>0,e_m[ie],1), w_n/e_n[ie])   # cuota MÍA en el gasto de la entidad

    hhi_p = np.bincount(ip, sh_p**2, nP)
    top_p = pd.Series(sh_p).groupby(ip).max().reindex(range(nP)).values
    cap_max = pd.Series(sh_e).groupby(ip).max().reindex(range(nP)).values      # máx. % del gasto de una entidad que capturo
    cap_med = np.bincount(ip, sh_e*sh_p, nP)                                    # captura media ponderada por mi cartera

    G = nx.Graph(); G.add_edges_from(zip(ip.tolist(), (ie+nP).tolist()))
    core = nx.core_number(G)                                                    # k-core de la red bipartita
    core_p = np.array([core.get(i,0) for i in range(nP)]); core_e = np.array([core.get(nP+j,0) for j in range(nE)])

    B = sp.csr_matrix((np.log1p(w_m), (ip, ie)), shape=(nP, nE))               # autovector bipartito (SVD)
    try:
        u, s, vt = svds(B, k=1, tol=1e-4); eig_p = np.abs(u[:,0]); eig_e = np.abs(vt[0,:])
    except Exception as ex:
        eig_p = np.full(nP, np.nan); eig_e = np.full(nE, np.nan); print('svds falló', Y, ex)

    filas_p.append(pd.DataFrame({'proveedor_ruc': pro_ids, 'anio_aplicable': Y,
        'red_n_entidades': p_deg, 'red_n_contratos': p_n, 'red_monto_total': p_m,
        'red_hhi_entidades': hhi_p, 'red_cuota_top_entidad': top_p,
        'red_captura_max': cap_max, 'red_captura_media': cap_med,
        'red_coreness': core_p, 'red_autovector': eig_p}))
    p_share_ent = pd.Series(sh_e).groupby(ie).max().reindex(range(nE)).values
    hhi_e = np.bincount(ie, sh_e**2, nE)
    filas_e.append(pd.DataFrame({'entidad_ruc': ent_ids, 'anio_aplicable': Y,
        'ent_n_proveedores': e_deg, 'ent_n_contratos': e_n, 'ent_monto_total': e_m,
        'ent_hhi_proveedores': hhi_e, 'ent_cuota_top_proveedor': p_share_ent,
        'ent_coreness': core_e, 'ent_autovector': eig_e}))
    resumen.append((Y, nP, nE, len(E), max(core.values()) if core else 0))
    print(f'  año {Y}: {nP:,} proveedores | {nE:,} entidades | {len(E):,} aristas | k-core máx {resumen[-1][-1]} | {time.time()-t0:.0f}s')

RP = pd.concat(filas_p, ignore_index=True); RE = pd.concat(filas_e, ignore_index=True)
for c in ['red_n_entidades','red_monto_total','red_cuota_top_entidad','red_captura_max','red_coreness','red_autovector']:
    RP[c+'_pct'] = RP.groupby('anio_aplicable')[c].rank(pct=True)              # percentil dentro del año

def guardar(df, nombre):
    tmp = f'{R}/{nombre}.tmp.parquet'
    df.to_parquet(tmp, index=False); os.replace(tmp, f'{R}/{nombre}.parquet')
guardar(RP, 'red_proveedor_anio'); guardar(RE, 'red_entidad_anio')
print(f'\nGuardado red_proveedor_anio ({len(RP):,} filas) y red_entidad_anio ({len(RE):,} filas)')

# 2) Cobertura (2015 = 0% por diseño: no hay año previo)
cob = con.execute(f"""
    SELECT b.anio_proceso AS anio, COUNT(*) AS filas,
           ROUND(100.0*AVG(CASE WHEN r.proveedor_ruc IS NOT NULL THEN 1 ELSE 0 END),1) AS pct_con_historial_red
    FROM bm3 b LEFT JOIN read_parquet('{R}/red_proveedor_anio.parquet') r
      ON b.proveedor_ruc=r.proveedor_ruc AND b.anio_proceso=r.anio_aplicable
    GROUP BY 1 ORDER BY 1""").df()
print('\nCobertura de historial de red por año:'); print(cob.to_string(index=False))

# 3) Diagnóstico con fantasmas (SOLO documentación)
d = con.execute("SELECT DISTINCT proveedor_ruc, anio_proceso, proveedor_es_fantasma AS f FROM bm3").df()
d = d.merge(RP, left_on=['proveedor_ruc','anio_proceso'], right_on=['proveedor_ruc','anio_aplicable'], how='inner')
print(f'\nDiagnóstico (proveedor-año con historial): {len(d):,} filas | fantasma: {int(d.f.sum())} filas, {d[d.f].proveedor_ruc.nunique()} RUC')
if d.f.sum() > 0:
    out = [(c.replace('_pct',''), d.loc[d.f, c].median(), d.loc[~d.f, c].median()) for c in d.columns if c.endswith('_pct')]
    print(pd.DataFrame(out, columns=['feature','mediana_pct_fantasma','mediana_pct_resto']).round(3).to_string(index=False))
    print('(n de fantasmas muy pequeño: orientativo, NO evidencia)')

In [ ]:
# ===== CAPA 2 · PARTE B1 · Red de co-participación entre oferentes (sin fuga) =====
import os, time, gc, glob, warnings
import numpy as np, pandas as pd, duckdb
import scipy.sparse as sp
import networkx as nx
warnings.filterwarnings('ignore')

os.chdir(r'C:\SERCOP_FINAL')
R = '03_resultados'
PARTES = f'{R}/red_cobid_partes'
os.makedirs(PARTES, exist_ok=True)
try:
    con
except NameError:
    con = duckdb.connect(); con.execute("PRAGMA threads=8")
con.execute(f"CREATE OR REPLACE VIEW bm3 AS SELECT * FROM read_parquet('{R}/base_maestra_v3.parquet')")

MAX_OFERENTES = 40   # procesos con más oferentes se excluyen de la red (generarían miles de pares)
MIN_PESO      = 2    # para núcleo/agrupamiento: solo pares que compitieron 2+ veces

# ---------- 1) Detectar columnas del archivo de oferentes ----------
ARCH_OF = f'{R}/oferentes_consolidado.parquet'
of_cols = [c[0] for c in con.execute(f"DESCRIBE SELECT * FROM read_parquet('{ARCH_OF}')").fetchall()]
print('Columnas de oferentes_consolidado:', of_cols)
c_ruc  = next((c for c in ['oferente_ruc','proveedor_ruc','ruc','RUC'] if c in of_cols), None)
c_ocid = next((c for c in ['ocid','OCID','id_proceso','codigo_proceso'] if c in of_cols), None)
c_anio = next((c for c in ['anio_proceso','anio','anio_release','year'] if c in of_cols), None)
assert c_ruc and c_ocid, f'No identifico columnas RUC/ocid en {of_cols}'
print(f'Usando: ocid={c_ocid} | ruc={c_ruc} | anio={c_anio or "(se toma de bm3 por ocid)"}')

anio_sql = f'o."{c_anio}"' if c_anio else 'b.anio_proceso'
join_sql = '' if c_anio else f'JOIN (SELECT DISTINCT ocid, anio_proceso FROM bm3) b ON o."{c_ocid}" = b.ocid'

pares_proc = con.execute(f"""
    SELECT DISTINCT o."{c_ocid}" AS ocid, o."{c_ruc}" AS ruc, {anio_sql} AS anio
    FROM read_parquet('{ARCH_OF}') o {join_sql}
    WHERE o."{c_ruc}" IS NOT NULL AND {anio_sql} IS NOT NULL
""").df()
tam = pares_proc.groupby('ocid')['ruc'].transform('size')
print(f'\nFilas oferente-proceso: {len(pares_proc):,} | procesos: {pares_proc.ocid.nunique():,}')
print('Oferentes por proceso:',
      pd.Series(tam.groupby(pares_proc.ocid).first()).describe()[['mean','50%','max']].round(1).to_dict())
excl = (tam > MAX_OFERENTES).sum()
print(f'Filas excluidas por proceso con >{MAX_OFERENTES} oferentes: {excl:,} ({100*excl/len(pares_proc):.2f}%)')
pares_proc = pares_proc[(tam >= 2) & (tam <= MAX_OFERENTES)].copy()
anios = sorted(pares_proc['anio'].unique())
print(f'Filas útiles para la red: {len(pares_proc):,} | años {anios[0]}-{anios[-1]}')

# ---------- 2) Triángulos por bloques (memoria controlada) ----------
def triangulos(P, bloque=4000):
    n = P.shape[0]; t = np.zeros(n)
    for i0 in range(0, n, bloque):
        i1 = min(i0+bloque, n)
        t[i0:i1] = np.asarray(((P[i0:i1] @ P).multiply(P[i0:i1])).sum(axis=1)).ravel()
    return t/2.0

# ---------- 3) Por año, usando SOLO años anteriores ----------
t0 = time.time()
for Y in anios:
    destino = f'{PARTES}/cobid_{Y}.parquet'
    if os.path.exists(destino):
        print(f'  año {Y}: ya existe, se salta'); continue
    E = pares_proc[pares_proc['anio'] < Y]
    if E.empty:
        print(f'  año {Y}: sin historial previo'); continue
    ruc_ids = pd.Index(E['ruc'].unique()); oc_ids = pd.Index(E['ocid'].unique())
    ir = ruc_ids.get_indexer(E['ruc']); io = oc_ids.get_indexer(E['ocid'])
    nR, nO = len(ruc_ids), len(oc_ids)
    A = sp.csr_matrix((np.ones(len(E)), (io, ir)), shape=(nO, nR))   # procesos x proveedores

    C = (A.T @ A).tocsr()                     # veces que i y j estuvieron en el mismo proceso
    n_proc = C.diagonal().copy()              # procesos en que participó i
    C.setdiag(0); C.eliminate_zeros()
    grado = np.diff(C.indptr).astype(float)   # rivales distintos
    peso  = np.asarray(C.sum(axis=1)).ravel()
    top_riv = np.zeros(nR); hhi_riv = np.zeros(nR)
    dat, ind = C.data, C.indptr
    for i in range(nR):                       # concentración de rivales
        a, b = ind[i], ind[i+1]
        if b > a:
            v = dat[a:b]; s = v.sum()
            top_riv[i] = v.max()/s; hhi_riv[i] = float(np.sum((v/s)**2))
    Cr = C.copy(); Cr.data = (Cr.data >= MIN_PESO).astype(float); Cr.eliminate_zeros()
    rec = np.diff(Cr.indptr).astype(float)    # rivales recurrentes

    print(f'  año {Y}: {nR:,} proveedores | red completa {C.nnz//2:,} pares | red recurrente {Cr.nnz//2:,} pares', end='')
    G = nx.from_scipy_sparse_array(Cr)
    core = nx.core_number(G); coren = np.array([core.get(i,0) for i in range(nR)], float)
    tri = triangulos(Cr)
    posibles = rec*(rec-1)/2.0
    clust = np.where(posibles > 0, tri/np.maximum(posibles,1), np.nan)

    pd.DataFrame({'proveedor_ruc': ruc_ids, 'anio_aplicable': Y,
        'cob_n_procesos': n_proc, 'cob_n_rivales': grado, 'cob_peso_rivales': peso,
        'cob_rivales_recurrentes': rec, 'cob_cuota_rival_top': top_riv, 'cob_hhi_rivales': hhi_riv,
        'cob_coreness': coren, 'cob_agrupamiento': clust,
        'cob_rivales_x_proceso': np.where(n_proc>0, grado/np.maximum(n_proc,1), np.nan)
    }).to_parquet(destino, index=False)
    print(f' | {time.time()-t0:.0f}s')
    del A, C, Cr, G; gc.collect()

# ---------- 4) Consolidar ----------
partes = sorted(glob.glob(f'{PARTES}/cobid_*.parquet'))
CB = pd.concat([pd.read_parquet(p) for p in partes], ignore_index=True)
for c in ['cob_n_rivales','cob_rivales_recurrentes','cob_cuota_rival_top','cob_hhi_rivales',
          'cob_coreness','cob_agrupamiento','cob_rivales_x_proceso']:
    CB[c+'_pct'] = CB.groupby('anio_aplicable')[c].rank(pct=True)
tmp = f'{R}/red_cobid_anio.tmp.parquet'
CB.to_parquet(tmp, index=False); os.replace(tmp, f'{R}/red_cobid_anio.parquet')
print(f'\nGuardado red_cobid_anio: {len(CB):,} filas | {CB.anio_aplicable.nunique()} años')

cob = con.execute(f"""
    SELECT b.anio_proceso AS anio, COUNT(*) AS filas,
           ROUND(100.0*AVG(CASE WHEN r.proveedor_ruc IS NOT NULL THEN 1 ELSE 0 END),1) AS pct_con_red_cobid
    FROM bm3 b LEFT JOIN read_parquet('{R}/red_cobid_anio.parquet') r
      ON b.proveedor_ruc=r.proveedor_ruc AND b.anio_proceso=r.anio_aplicable
    GROUP BY 1 ORDER BY 1""").df()
print('\nCobertura de la red de co-participación por año:'); print(cob.to_string(index=False))
print('\nResumen de las variables nuevas (último año disponible):')
ult = CB[CB.anio_aplicable == CB.anio_aplicable.max()]
print(ult[[c for c in CB.columns if c.startswith('cob_') and not c.endswith('_pct')]].describe().T[['mean','50%','max']].round(3).to_string())

In [ ]:
# ===== CAPA 2 · PARTE B2 · Fantasmas CONTROLANDO TAMAÑO (solo documentación) =====
import os, numpy as np, pandas as pd, duckdb, warnings
warnings.filterwarnings('ignore')
os.chdir(r'C:\SERCOP_FINAL'); R = '03_resultados'
try: con
except NameError:
    con = duckdb.connect(); con.execute("PRAGMA threads=8")
con.execute(f"CREATE OR REPLACE VIEW bm3 AS SELECT * FROM read_parquet('{R}/base_maestra_v3.parquet')")

pan = con.execute("""SELECT proveedor_ruc, anio_proceso AS anio,
                            MAX(CASE WHEN proveedor_es_fantasma THEN 1 ELSE 0 END) AS f
                     FROM bm3 GROUP BY 1,2""").df()
RP = pd.read_parquet(f'{R}/red_proveedor_anio.parquet')
CB = pd.read_parquet(f'{R}/red_cobid_anio.parquet')
d = (pan.merge(RP, left_on=['proveedor_ruc','anio'], right_on=['proveedor_ruc','anio_aplicable'], how='inner')
        .merge(CB, left_on=['proveedor_ruc','anio'], right_on=['proveedor_ruc','anio_aplicable'],
               how='left', suffixes=('','_cb')))
feats = [c for c in d.columns if (c.startswith('red_') or c.startswith('cob_')) and not c.endswith('_pct')]
print(f'Panel proveedor-año con historial de red: {len(d):,} filas')
print(f'Fantasmas: {int(d.f.sum())} filas proveedor-año | {d.loc[d.f==1,"proveedor_ruc"].nunique()} RUC')

# Estrato = año x decil de actividad previa
d['tam'] = d['red_n_contratos'].fillna(0)
d['decil'] = d.groupby('anio')['tam'].transform(lambda s: pd.qcut(s.rank(method='first'), 10, labels=False, duplicates='drop'))
d['estrato'] = d['anio'].astype(str) + '_' + d['decil'].astype(str)

print('\nDistribución de fantasmas por decil de actividad (1=menos activo, 10=más):')
t = d[d.f==1].groupby('decil').size().reindex(range(10), fill_value=0)
print(pd.DataFrame({'decil': range(1,11), 'filas_fantasma': t.values}).to_string(index=False))

filas = []
for c in feats:
    sub = d[['estrato','f',c]].dropna()
    sub['p'] = sub.groupby('estrato')[c].rank(pct=True)   # percentil DENTRO del estrato
    fa = sub.loc[sub.f==1,'p']
    if len(fa) >= 5:
        filas.append((c, len(fa), round(fa.median(),3), round(float((fa>0.5).mean()),3)))
res = pd.DataFrame(filas, columns=['variable','n_fantasma','mediana_percentil_intra_estrato','prop_sobre_la_mediana'])
res = res.sort_values('mediana_percentil_intra_estrato', ascending=False)
print('\nFantasmas vs proveedores del MISMO año y MISMO decil de actividad')
print('(0.50 = igual al grupo de comparación; >0.50 = el fantasma puntúa más alto)')
print(res.to_string(index=False))
print('\nOJO: con este número de fantasmas es orientativo, no reemplaza al modelo de la Capa 3.')

In [ ]:
# ===== CAPA 2 · PARTE B3 · Comunidades y rol estructural (Guimerà-Amaral) =====
import os, time, gc, glob, warnings
import numpy as np, pandas as pd, duckdb
import scipy.sparse as sp
import networkx as nx
warnings.filterwarnings('ignore')

os.chdir(r'C:\SERCOP_FINAL'); R = '03_resultados'
PARTES = f'{R}/red_comunidad_partes'; os.makedirs(PARTES, exist_ok=True)
try: con
except NameError:
    con = duckdb.connect(); con.execute("PRAGMA threads=8")
con.execute(f"CREATE OR REPLACE VIEW bm3 AS SELECT * FROM read_parquet('{R}/base_maestra_v3.parquet')")

MAX_OFERENTES = 40; MIN_PESO = 2; SEED = 7

pares_proc = con.execute(f"""
    SELECT DISTINCT ocid, oferente_ruc AS ruc, anio
    FROM read_parquet('{R}/oferentes_consolidado.parquet')
    WHERE TRIM(COALESCE(oferente_ruc,'')) <> '' AND anio IS NOT NULL
""").df()
tam = pares_proc.groupby('ocid')['ruc'].transform('size')
pares_proc = pares_proc[(tam >= 2) & (tam <= MAX_OFERENTES)].copy()
anios = sorted(pares_proc['anio'].unique())
print(f'Filas útiles: {len(pares_proc):,} | años {anios[0]}-{anios[-1]}')

def rol_guimera(z, P, iso):
    r = np.full(len(z), 'R0_sin_rival_recurrente', dtype=object)
    ok = ~iso; nh = ok & (z < 2.5); h = ok & (z >= 2.5)
    r[nh & (P <= 0.05)]              = 'R1_ultraperiferico'
    r[nh & (P > 0.05) & (P <= 0.62)] = 'R2_periferico'
    r[nh & (P > 0.62) & (P <= 0.80)] = 'R3_conector'
    r[nh & (P > 0.80)]               = 'R4_sin_clan'
    r[h  & (P <= 0.30)]              = 'R5_hub_provincial'
    r[h  & (P > 0.30) & (P <= 0.75)] = 'R6_hub_conector'
    r[h  & (P > 0.75)]               = 'R7_hub_sin_clan'
    return r

t0 = time.time()
for Y in anios:
    destino = f'{PARTES}/com_{Y}.parquet'
    if os.path.exists(destino):
        print(f'  año {Y}: ya existe, se salta'); continue
    E = pares_proc[pares_proc['anio'] < Y]
    if E.empty:
        print(f'  año {Y}: sin historial previo'); continue
    ruc_ids = pd.Index(E['ruc'].unique()); oc_ids = pd.Index(E['ocid'].unique())
    ir = ruc_ids.get_indexer(E['ruc']); io = oc_ids.get_indexer(E['ocid'])
    nR = len(ruc_ids)
    A = sp.csr_matrix((np.ones(len(E)), (io, ir)), shape=(len(oc_ids), nR))
    C = (A.T @ A).tocsr(); C.setdiag(0); C.eliminate_zeros()
    W = C.copy(); W.data = np.where(W.data >= MIN_PESO, W.data, 0.0); W.eliminate_zeros()
    del A; gc.collect()

    k_tot = np.asarray(W.sum(axis=1)).ravel()         # grado pesado en la red recurrente
    k_full = np.asarray(C.sum(axis=1)).ravel()        # grado pesado en la red completa
    iso   = k_tot == 0                                # sin ningún rival recurrente
    idx   = np.where(~iso)[0]
    Ws    = W[idx][:, idx]

    G = nx.from_scipy_sparse_array(Ws)
    com = nx.community.louvain_communities(G, weight='weight', seed=SEED)
    lab_s = np.empty(len(idx), dtype=np.int64)
    for k, cset in enumerate(com):
        lab_s[list(cset)] = k
    nC = len(com)
    lab = np.full(nR, -1, dtype=np.int64); lab[idx] = lab_s

    S = sp.csr_matrix((np.ones(len(idx)), (idx, lab_s)), shape=(nR, nC))
    def participacion(X, k):
        M = (X @ S).tocsr()
        inv = np.zeros(nR); ok = k > 0; inv[ok] = 1.0/k[ok]
        F = M.multiply(inv[:, None]).tocsr()
        p = 1.0 - np.asarray(F.multiply(F).sum(axis=1)).ravel()
        p[~ok] = np.nan
        return M, p
    M, P = participacion(W, k_tot)                    # participación en la red recurrente
    _, P_full = participacion(C, k_full)              # participación contando TODA la competencia
    P[iso] = np.nan; P_full[iso] = np.nan
    k_in = np.asarray(M.multiply(S).sum(axis=1)).ravel()
    n_com_vec = np.diff(M.indptr).astype(float); n_com_vec[iso] = np.nan

    dz = pd.DataFrame({'lab': lab[~iso], 'k_in': k_in[~iso]})
    mu = dz.groupby('lab')['k_in'].transform('mean'); sd = dz.groupby('lab')['k_in'].transform('std')
    z = np.full(nR, np.nan); z[~iso] = ((dz['k_in']-mu)/sd.replace(0, np.nan)).fillna(0.0).values
    tam_com = np.full(nR, np.nan); vc = pd.Series(lab_s).value_counts()
    tam_com[~iso] = pd.Series(lab_s).map(vc).values

    pd.DataFrame({'proveedor_ruc': ruc_ids, 'anio_aplicable': Y,
        'com_id': lab, 'com_tamano': tam_com,
        'com_participacion': P, 'com_participacion_total': P_full,
        'com_z_intra': z, 'com_n_comunidades_rivales': n_com_vec,
        'com_grado_intra': k_in, 'com_grado_total': k_tot,
        'com_grado_completo': k_full,
        'com_rol': rol_guimera(np.nan_to_num(z), np.nan_to_num(P), iso)
    }).to_parquet(destino, index=False)
    print(f'  año {Y}: {nR:,} proveedores | {int((~iso).sum()):,} en red recurrente | '
          f'{nC:,} comunidades | mayor {int(vc.max()):,} | {time.time()-t0:.0f}s')
    del W, C, Ws, G, S, M; gc.collect()

partes = sorted(glob.glob(f'{PARTES}/com_*.parquet'))
CM = pd.concat([pd.read_parquet(p) for p in partes], ignore_index=True)
for c in ['com_participacion','com_participacion_total','com_z_intra','com_n_comunidades_rivales','com_tamano']:
    CM[c+'_pct'] = CM.groupby('anio_aplicable')[c].rank(pct=True)
tmp = f'{R}/red_comunidad_anio.tmp.parquet'
CM.to_parquet(tmp, index=False); os.replace(tmp, f'{R}/red_comunidad_anio.parquet')
print(f'\nGuardado red_comunidad_anio: {len(CM):,} filas')

print('\nDistribución de roles estructurales (último año, %):')
u = CM[CM.anio_aplicable == CM.anio_aplicable.max()]
print((u['com_rol'].value_counts(normalize=True)*100).round(2).to_string())

pan = con.execute("""SELECT proveedor_ruc, anio_proceso AS anio,
                            MAX(CASE WHEN proveedor_es_fantasma THEN 1 ELSE 0 END) AS f
                     FROM bm3 GROUP BY 1,2""").df()
dd = pan.merge(CM, left_on=['proveedor_ruc','anio'], right_on=['proveedor_ruc','anio_aplicable'], how='inner')
print(f'\nPanel con comunidad: {len(dd):,} | fantasmas: {int(dd.f.sum())} filas, {dd.loc[dd.f==1,"proveedor_ruc"].nunique()} RUC')
if dd.f.sum() > 0:
    comp = pd.concat([(dd[dd.f==1]['com_rol'].value_counts(normalize=True)*100).rename('fantasma_%'),
                      (dd[dd.f==0]['com_rol'].value_counts(normalize=True)*100).rename('resto_%')], axis=1).fillna(0).round(1)
    print('\nRol estructural: fantasma vs resto'); print(comp.to_string())
    print('\nMediana de percentil dentro del año (fantasma | resto):')
    for c in ['com_participacion_pct','com_participacion_total_pct','com_z_intra_pct','com_n_comunidades_rivales_pct','com_tamano_pct']:
        print(f'  {c}: {dd.loc[dd.f==1,c].median():.3f} | {dd.loc[dd.f==0,c].median():.3f}')

In [ ]:
print([c[0] for c in con.execute("DESCRIBE SELECT * FROM read_parquet('03_resultados/pares_mismo_representante.parquet')").fetchall()])

In [ ]:
# ===== CAPA 2 · PARTE B4 · Prueba de permutación por conglomerado (RUC) =====
import os, numpy as np, pandas as pd, duckdb, warnings
warnings.filterwarnings('ignore')
os.chdir(r'C:\SERCOP_FINAL'); R = '03_resultados'
try: con
except NameError:
    con = duckdb.connect(); con.execute("PRAGMA threads=8")
con.execute(f"CREATE OR REPLACE VIEW bm3 AS SELECT * FROM read_parquet('{R}/base_maestra_v3.parquet')")

B = 10000; rng = np.random.default_rng(20260930)

pan = con.execute("""SELECT proveedor_ruc, anio_proceso AS anio,
                            MAX(CASE WHEN proveedor_es_fantasma THEN 1 ELSE 0 END) AS f
                     FROM bm3 GROUP BY 1,2""").df()
RP = pd.read_parquet(f'{R}/red_proveedor_anio.parquet')
CB = pd.read_parquet(f'{R}/red_cobid_anio.parquet')
CM = pd.read_parquet(f'{R}/red_comunidad_anio.parquet')
d = pan.merge(RP, left_on=['proveedor_ruc','anio'], right_on=['proveedor_ruc','anio_aplicable'], how='inner')
for T in (CB, CM):
    d = d.merge(T, left_on=['proveedor_ruc','anio'], right_on=['proveedor_ruc','anio_aplicable'],
                how='left', suffixes=('', '_x'))
    d = d.drop(columns=[c for c in d.columns if c.endswith('_x')])
feats = [c for c in d.columns
         if (c.startswith('red_') or c.startswith('cob_') or c.startswith('com_'))
         and not c.endswith('_pct') and c not in ('com_id','com_rol')
         and pd.api.types.is_numeric_dtype(d[c])]
print(f'Panel: {len(d):,} filas | {d.proveedor_ruc.nunique():,} RUC | {len(feats)} variables')

d['tam'] = d['red_n_contratos'].fillna(0)
d['decil'] = d.groupby('anio')['tam'].transform(lambda s: pd.qcut(s.rank(method='first'), 10, labels=False, duplicates='drop'))
d['estrato'] = d['anio'].astype(str) + '_' + d['decil'].astype(str)

# percentiles DENTRO del estrato: no dependen de la etiqueta, se calculan una sola vez
Pm = np.full((len(d), len(feats)), np.nan)
for j, c in enumerate(feats):
    Pm[:, j] = d.groupby('estrato')[c].rank(pct=True).values

ruc_tam = d.groupby('proveedor_ruc')['tam'].max()
ruc_f   = d.groupby('proveedor_ruc')['f'].max()
ruc_dec = pd.qcut(ruc_tam.rank(method='first'), 10, labels=False, duplicates='drop')
fant = ruc_f[ruc_f == 1].index.tolist()
print(f'RUC fantasma con historial: {len(fant)}')
pos = d.groupby('proveedor_ruc').indices          # posiciones por RUC
obs_idx = np.concatenate([pos[r] for r in fant])
obs = np.nanmedian(Pm[obs_idx], axis=0)

cand = {dc: np.array([r for r in ruc_dec[ruc_dec == dc].index if ruc_f[r] == 0], dtype=object)
        for dc in sorted(ruc_dec.dropna().unique())}
dec_f = [ruc_dec[r] for r in fant]
print('Deciles de tamaño de los RUC fantasma:', sorted(int(x) for x in dec_f))

nul = np.full((B, len(feats)), np.nan)
for b in range(B):
    sel = [rng.choice(cand[dc]) for dc in dec_f]
    idx = np.concatenate([pos[r] for r in sel])
    nul[b] = np.nanmedian(Pm[idx], axis=0)

p_raw = (1 + np.nansum(np.abs(nul - 0.5) >= np.abs(obs - 0.5)[None, :], axis=0)) / (B + 1)
o = np.argsort(p_raw); m = len(p_raw)          # Benjamini-Hochberg
q = np.empty(m); acc = 1.0
for k in range(m-1, -1, -1):
    acc = min(acc, p_raw[o[k]] * m / (k+1)); q[o[k]] = acc

res = pd.DataFrame({'variable': feats, 'mediana_fantasma': obs.round(3),
                    'nulo_medio': np.nanmean(nul, axis=0).round(3),
                    'nulo_p2.5': np.nanpercentile(nul, 2.5, axis=0).round(3),
                    'nulo_p97.5': np.nanpercentile(nul, 97.5, axis=0).round(3),
                    'p': p_raw.round(4), 'q_BH': q.round(4)})
res['signif'] = np.where(res.q_BH < 0.05, '**', np.where(res.p < 0.05, '*(sin corregir)', ''))
res = res.reindex(res.mediana_fantasma.sub(0.5).abs().sort_values(ascending=False).index)
print(f'\nPrueba de permutación por conglomerado, B={B:,}')
print(res.to_string(index=False))
print(f'\nSignificativas tras Benjamini-Hochberg (q<0.05): {int((res.q_BH<0.05).sum())} de {m}')

In [ ]:
# ===== CAPA 2 · PARTE C · Grupos por representante legal y competencia intragrupo =====
import os, warnings
import numpy as np, pandas as pd, duckdb
import scipy.sparse as sp
from scipy.sparse.csgraph import connected_components
warnings.filterwarnings('ignore')

os.chdir(r'C:\SERCOP_FINAL'); R = '03_resultados'
try: con
except NameError:
    con = duckdb.connect(); con.execute("PRAGMA threads=8")
con.execute(f"CREATE OR REPLACE VIEW bm3 AS SELECT * FROM read_parquet('{R}/base_maestra_v3.parquet')")

MAX_OFERENTES = 40
TAM_GRUPO_MAX = 50     # grupos mayores se marcan como representante nominal, no como grupo real

# ---------- 1) Grupos = componentes conexas del grafo de mismo representante ----------
pr = pd.read_parquet(f'{R}/pares_mismo_representante.parquet')
print(f'Pares mismo representante: {len(pr):,}')
print(pr[['coincidencias','dominancia_max']].describe().T[['mean','50%','max']].round(2).to_string())
rucs = pd.Index(pd.unique(pd.concat([pr.ruc_a, pr.ruc_b], ignore_index=True)))
ia = rucs.get_indexer(pr.ruc_a); ib = rucs.get_indexer(pr.ruc_b)
n = len(rucs)
Gm = sp.coo_matrix((np.ones(len(pr)), (ia, ib)), shape=(n, n))
ncomp, lab = connected_components(Gm, directed=False)
grp = pd.DataFrame({'proveedor_ruc': rucs, 'grupo_id': lab})
grp['grupo_tamano'] = grp.groupby('grupo_id')['proveedor_ruc'].transform('size')
print(f'\nRUC en algún grupo: {n:,} | grupos: {ncomp:,}')
print('Tamaño de grupo:', grp.grupo_tamano.describe()[['mean','50%','max']].round(1).to_dict())
print('\nGrupos más grandes (revisar si el representante es nominal o un estudio jurídico):')
top = grp.groupby('grupo_id')['grupo_tamano'].first().sort_values(ascending=False).head(8)
for gid, t in top.items():
    ej = pr[(pr.ruc_a.isin(grp[grp.grupo_id==gid].proveedor_ruc)) | (pr.ruc_b.isin(grp[grp.grupo_id==gid].proveedor_ruc))]
    rep = pd.concat([ej.representante_a, ej.representante_b]).value_counts().head(1)
    print(f'  grupo {gid}: {t} empresas | representante más frecuente: {rep.index[0] if len(rep) else "?"} ({rep.iloc[0] if len(rep) else 0} pares)')
grp['grupo_sospechoso_nominal'] = grp.grupo_tamano > TAM_GRUPO_MAX
print(f'\nRUC en grupos de más de {TAM_GRUPO_MAX} empresas: {int(grp.grupo_sospechoso_nominal.sum()):,}')

# ---------- 2) Competencia intragrupo, con ventana expansiva ----------
of = con.execute(f"""
    SELECT DISTINCT ocid, oferente_ruc AS ruc, anio
    FROM read_parquet('{R}/oferentes_consolidado.parquet')
    WHERE TRIM(COALESCE(oferente_ruc,'')) <> '' AND anio IS NOT NULL
""").df()
tam = of.groupby('ocid')['ruc'].transform('size')
of = of[(tam >= 2) & (tam <= MAX_OFERENTES)].copy()
of = of.merge(grp[['proveedor_ruc','grupo_id','grupo_tamano']], left_on='ruc', right_on='proveedor_ruc', how='left')
en_grupo = of.grupo_id.notna()
print(f'\nFilas oferente-proceso: {len(of):,} | con grupo identificado: {int(en_grupo.sum()):,} ({100*en_grupo.mean():.1f}%)')

g = of[en_grupo & (of.grupo_tamano <= TAM_GRUPO_MAX)]
cnt = g.groupby(['ocid','grupo_id'])['ruc'].nunique().rename('n_del_grupo').reset_index()
g = g.merge(cnt, on=['ocid','grupo_id'])
g['intragrupo'] = (g.n_del_grupo >= 2).astype(int)
print(f'Procesos con competencia intragrupo: {g.loc[g.intragrupo==1,"ocid"].nunique():,} '
      f'de {g.ocid.nunique():,} procesos con empresas agrupadas')

por_anio = g.groupby(['ruc','anio']).agg(proc=('ocid','nunique'), intra=('intragrupo','sum')).reset_index()
filas = []
for Y in sorted(of.anio.unique()):
    h = por_anio[por_anio.anio < Y]
    if h.empty: continue
    a = h.groupby('ruc').agg(rep_proc_previos=('proc','sum'), rep_proc_intragrupo=('intra','sum')).reset_index()
    a['anio_aplicable'] = Y
    a['rep_tasa_intragrupo'] = a.rep_proc_intragrupo / a.rep_proc_previos.replace(0, np.nan)
    filas.append(a)
RG = pd.concat(filas, ignore_index=True).rename(columns={'ruc':'proveedor_ruc'})
RG = RG.merge(grp[['proveedor_ruc','grupo_id','grupo_tamano','grupo_sospechoso_nominal']], on='proveedor_ruc', how='left')
for c in ['rep_proc_intragrupo','rep_tasa_intragrupo','grupo_tamano']:
    RG[c+'_pct'] = RG.groupby('anio_aplicable')[c].rank(pct=True)
tmp = f'{R}/red_representante_anio.tmp.parquet'
RG.to_parquet(tmp, index=False); os.replace(tmp, f'{R}/red_representante_anio.parquet')
print(f'\nGuardado red_representante_anio: {len(RG):,} filas')
print(RG[['rep_proc_previos','rep_proc_intragrupo','rep_tasa_intragrupo','grupo_tamano']].describe().T[['mean','50%','max']].round(3).to_string())

# ---------- 3) Los fantasmas ----------
pan = con.execute("""SELECT proveedor_ruc, anio_proceso AS anio,
                            MAX(CASE WHEN proveedor_es_fantasma THEN 1 ELSE 0 END) AS f
                     FROM bm3 GROUP BY 1,2""").df()
fr = pan[pan.f==1].merge(grp, on='proveedor_ruc', how='left')
print(f'\nRUC fantasma: {pan[pan.f==1].proveedor_ruc.nunique()} | con grupo identificado: {fr[fr.grupo_id.notna()].proveedor_ruc.nunique()}')
dd = pan.merge(RG, left_on=['proveedor_ruc','anio'], right_on=['proveedor_ruc','anio_aplicable'], how='inner')
if dd.f.sum() > 0:
    print(f'Panel con datos de grupo: {len(dd):,} | fantasmas: {int(dd.f.sum())} filas')
    print('\nMediana de percentil dentro del año (fantasma | resto):')
    for c in ['rep_proc_intragrupo_pct','rep_tasa_intragrupo_pct','grupo_tamano_pct']:
        print(f'  {c}: {dd.loc[dd.f==1,c].median():.3f} | {dd.loc[dd.f==0,c].median():.3f}')
    print('\nTasa de competencia intragrupo (cruda):')
    print(f'  fantasma: {dd.loc[dd.f==1,"rep_tasa_intragrupo"].mean():.4f} | resto: {dd.loc[dd.f==0,"rep_tasa_intragrupo"].mean():.4f}')

In [ ]:
# ===== CAPA 2 · PARTE B5 · Dos índices teóricos y UNA prueba para cada uno =====
import os, numpy as np, pandas as pd, duckdb, warnings
warnings.filterwarnings('ignore')
os.chdir(r'C:\SERCOP_FINAL'); R = '03_resultados'
try: con
except NameError:
    con = duckdb.connect(); con.execute("PRAGMA threads=8")
con.execute(f"CREATE OR REPLACE VIEW bm3 AS SELECT * FROM read_parquet('{R}/base_maestra_v3.parquet')")
B = 20000; rng = np.random.default_rng(20260930)

pan = con.execute("""SELECT proveedor_ruc, anio_proceso AS anio,
                            MAX(CASE WHEN proveedor_es_fantasma THEN 1 ELSE 0 END) AS f
                     FROM bm3 GROUP BY 1,2""").df()
d = pan.merge(pd.read_parquet(f'{R}/red_proveedor_anio.parquet'),
              left_on=['proveedor_ruc','anio'], right_on=['proveedor_ruc','anio_aplicable'], how='inner')
for arch in ['red_cobid_anio.parquet','red_comunidad_anio.parquet']:
    T = pd.read_parquet(f'{R}/{arch}')
    d = d.merge(T, left_on=['proveedor_ruc','anio'], right_on=['proveedor_ruc','anio_aplicable'],
                how='left', suffixes=('', '_x'))
    d = d.drop(columns=[c for c in d.columns if c.endswith('_x')])

d['tam'] = d['red_n_contratos'].fillna(0)
d['decil'] = d.groupby('anio')['tam'].transform(lambda s: pd.qcut(s.rank(method='first'), 10, labels=False, duplicates='drop'))
d['estrato'] = d['anio'].astype(str) + '_' + d['decil'].astype(str)

# ---- Índices derivados de la teoría ----
# Intermediación (Burt, agujeros estructurales; Guimerà-Amaral, rol conector):
#   muchos compradores y rivales diversos, posición central, SIN grupo cerrado
IDX = {
 'intermediacion': {'red_autovector': +1, 'red_n_entidades': +1, 'cob_n_rivales': +1,
                    'com_n_comunidades_rivales': +1, 'com_participacion': +1,
                    'cob_agrupamiento': -1, 'red_hhi_entidades': -1, 'cob_hhi_rivales': -1},
 # Cartel (Wachs-Kertész): grupo cerrado, pocos rivales y siempre los mismos
 'cartel':         {'cob_agrupamiento': +1, 'cob_hhi_rivales': +1, 'cob_cuota_rival_top': +1,
                    'com_z_intra': +1, 'cob_n_rivales': -1, 'com_participacion': -1},
}
for nombre, comp in IDX.items():
    partes = []
    for c, signo in comp.items():
        p = d.groupby('estrato')[c].rank(pct=True)           # percentil dentro de año x decil
        partes.append(signo*p + (0.0 if signo > 0 else 1.0)) # el signo negativo invierte: 1-p
    M = pd.concat(partes, axis=1)
    d['idx_'+nombre] = M.mean(axis=1, skipna=True)           # promedia solo lo disponible, sin imputar
    d['n_comp_'+nombre] = M.notna().sum(axis=1)
    print(f'idx_{nombre}: {int(d["idx_"+nombre].notna().sum()):,} filas con valor | '
          f'componentes disponibles (mediana) {int(d["n_comp_"+nombre].median())} de {len(comp)}')

ruc_f = d.groupby('proveedor_ruc')['f'].max()
ruc_dec = pd.qcut(d.groupby('proveedor_ruc')['tam'].max().rank(method='first'), 10, labels=False, duplicates='drop')
fant = ruc_f[ruc_f == 1].index.tolist()
pos = d.groupby('proveedor_ruc').indices
cand = {dc: np.array([r for r in ruc_dec[ruc_dec == dc].index if ruc_f[r] == 0], dtype=object)
        for dc in sorted(ruc_dec.dropna().unique())}
dec_f = [ruc_dec[r] for r in fant]

nom = list(IDX)
Pm = np.column_stack([d.groupby('estrato')['idx_'+k].rank(pct=True).values for k in nom])
obs = np.nanmedian(Pm[np.concatenate([pos[r] for r in fant])], axis=0)
nul = np.full((B, len(nom)), np.nan)
for b in range(B):
    idx = np.concatenate([pos[rng.choice(cand[dc])] for dc in dec_f])
    nul[b] = np.nanmedian(Pm[idx], axis=0)
p = (1 + np.nansum(np.abs(nul-0.5) >= np.abs(obs-0.5)[None,:], axis=0)) / (B+1)
q = np.minimum(1, p*len(nom)/np.argsort(np.argsort(p)).clip(1))

print(f'\nPrueba única por índice, B={B:,}, conglomerado = RUC, {len(fant)} RUC fantasma')
print(pd.DataFrame({'indice': nom, 'mediana_fantasma': obs.round(3),
                    'nulo_p2.5': np.nanpercentile(nul,2.5,axis=0).round(3),
                    'nulo_p97.5': np.nanpercentile(nul,97.5,axis=0).round(3),
                    'p': p.round(4), 'q_BH_2pruebas': q.round(4)}).to_string(index=False))

print('\nDónde caen los fantasmas en el índice de intermediación:')
u = d[d['idx_intermediacion'].notna()]
pct = u.groupby('estrato')['idx_intermediacion'].rank(pct=True)
print(f'  percentil mediano fantasma: {pct[u.f==1].median():.3f} | resto: {pct[u.f==0].median():.3f}')
print(f'  fantasmas en el 10% superior: {int((pct[u.f==1]>0.9).sum())} de {int((u.f==1).sum())}')
print(f'  fantasmas en el 25% superior: {int((pct[u.f==1]>0.75).sum())} de {int((u.f==1).sum())}')
tmp = f'{R}/indices_red.tmp.parquet'
d[['proveedor_ruc','anio','idx_intermediacion','idx_cartel','n_comp_intermediacion','n_comp_cartel']].to_parquet(tmp, index=False)
os.replace(tmp, f'{R}/indices_red.parquet'); print('\nGuardado indices_red.parquet')

In [ ]:
print([c[0] for c in con.execute(
    "DESCRIBE SELECT * FROM read_parquet('03_resultados/directorio_companias.parquet')").fetchall()])

In [ ]:
# ===== CAPA 2 · PARTE C (v2) · Grupos por representante legal, desde el DIRECTORIO =====
import os, re, unicodedata, warnings
import numpy as np, pandas as pd, duckdb
warnings.filterwarnings('ignore')
os.chdir(r'C:\SERCOP_FINAL'); R = '03_resultados'
try: con
except NameError:
    con = duckdb.connect(); con.execute("PRAGMA threads=8")
con.execute(f"CREATE OR REPLACE VIEW bm3 AS SELECT * FROM read_parquet('{R}/base_maestra_v3.parquet')")

MAX_OFERENTES = 40
TAM_GRUPO_MAX = 30     # arriba de esto se asume representante nominal o estudio jurídico

# ---------- 1) Normalizar el nombre del representante ----------
dr = con.execute(f"""
    SELECT RUC, NOMBRE, REPRESENTANTE, CARGO, PROVINCIA, "SITUACIÓN LEGAL" AS situacion, TIPO,
           FECHA_CONSTITUCION
    FROM read_parquet('{R}/directorio_companias.parquet')
""").df().drop_duplicates(subset='RUC')
print(f'Directorio: {len(dr):,} RUC únicos')
print('Cargos más frecuentes:'); print(dr.CARGO.value_counts().head(6).to_string())

def norm(s):
    if pd.isna(s): return None
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii','ignore').decode('ascii').upper()
    s = re.sub(r'[^A-Z ]', ' ', s); s = re.sub(r'\s+', ' ', s).strip()
    return s if len(s) >= 8 and len(s.split()) >= 2 else None    # descarta vacíos y basura
dr['rep_norm'] = dr.REPRESENTANTE.map(norm)
sin = dr.rep_norm.isna().sum()
print(f'\nRUC sin representante utilizable: {sin:,} ({100*sin/len(dr):.1f}%)')

g = dr[dr.rep_norm.notna()].copy()
g['grupo_tamano'] = g.groupby('rep_norm')['RUC'].transform('size')
print(f'RUC con representante: {len(g):,} | representantes distintos: {g.rep_norm.nunique():,}')
print('Tamaño de grupo:', g.grupo_tamano.describe()[['mean','50%','max']].round(1).to_dict())
print('\nRepresentantes con más empresas (candidatos a nominal):')
print(g.groupby('rep_norm')['RUC'].size().sort_values(ascending=False).head(10).to_string())
g['grupo_nominal'] = g.grupo_tamano > TAM_GRUPO_MAX
print(f'\nRUC en grupos de más de {TAM_GRUPO_MAX}: {int(g.grupo_nominal.sum()):,} '
      f'({100*g.grupo_nominal.mean():.2f}%) -> se marcan y NO se tratan como grupo real')
print(f'RUC en grupos de 2 a {TAM_GRUPO_MAX} empresas: {int(((g.grupo_tamano>=2)&(~g.grupo_nominal)).sum()):,}')

grp = g[['RUC','rep_norm','grupo_tamano','grupo_nominal','PROVINCIA']].rename(columns={'RUC':'proveedor_ruc'})

# ---------- 2) Competencia intragrupo con ventana expansiva ----------
of = con.execute(f"""
    SELECT DISTINCT ocid, oferente_ruc AS ruc, anio
    FROM read_parquet('{R}/oferentes_consolidado.parquet')
    WHERE TRIM(COALESCE(oferente_ruc,'')) <> '' AND anio IS NOT NULL
""").df()
tam = of.groupby('ocid')['ruc'].transform('size')
of = of[(tam >= 2) & (tam <= MAX_OFERENTES)].copy()
of = of.merge(grp, left_on='ruc', right_on='proveedor_ruc', how='left')
print(f'\nFilas oferente-proceso: {len(of):,} | con representante: {int(of.rep_norm.notna().sum()):,} '
      f'({100*of.rep_norm.notna().mean():.1f}%)')

gg = of[of.rep_norm.notna() & (of.grupo_nominal == False) & (of.grupo_tamano >= 2)]
cnt = gg.groupby(['ocid','rep_norm'])['ruc'].nunique().rename('n_del_grupo').reset_index()
gg = gg.merge(cnt, on=['ocid','rep_norm'])
gg['intragrupo'] = (gg.n_del_grupo >= 2).astype(int)
n_i = gg.loc[gg.intragrupo==1,'ocid'].nunique()
print(f'Procesos con 2+ empresas del mismo representante compitiendo: {n_i:,} '
      f'de {gg.ocid.nunique():,} procesos con empresas agrupadas '
      f'({100*n_i/max(gg.ocid.nunique(),1):.2f}%)')
print(f'  sobre el total de procesos analizados: {100*n_i/of.ocid.nunique():.3f}%')

por = gg.groupby(['ruc','anio']).agg(proc=('ocid','nunique'), intra=('intragrupo','sum')).reset_index()
filas = []
for Y in sorted(of.anio.unique()):
    h = por[por.anio < Y]
    if h.empty: continue
    a = h.groupby('ruc').agg(rep_proc_previos=('proc','sum'), rep_proc_intragrupo=('intra','sum')).reset_index()
    a['anio_aplicable'] = Y
    a['rep_tasa_intragrupo'] = a.rep_proc_intragrupo / a.rep_proc_previos.replace(0, np.nan)
    filas.append(a)
RG = (pd.concat(filas, ignore_index=True).rename(columns={'ruc':'proveedor_ruc'})
        .merge(grp.drop(columns=['PROVINCIA']), on='proveedor_ruc', how='left'))
for c in ['rep_proc_intragrupo','rep_tasa_intragrupo','grupo_tamano']:
    RG[c+'_pct'] = RG.groupby('anio_aplicable')[c].rank(pct=True)
tmp = f'{R}/red_representante_anio.tmp.parquet'
RG.drop(columns=['rep_norm']).to_parquet(tmp, index=False)
os.replace(tmp, f'{R}/red_representante_anio.parquet')
print(f'\nGuardado red_representante_anio: {len(RG):,} filas | '
      f'{RG.proveedor_ruc.nunique():,} RUC | años {RG.anio_aplicable.min()}-{RG.anio_aplicable.max()}')
print(RG[['rep_proc_previos','rep_proc_intragrupo','rep_tasa_intragrupo','grupo_tamano']]
      .describe().T[['mean','50%','max']].round(3).to_string())

# ---------- 3) Fantasmas (documentación) ----------
pan = con.execute("""SELECT proveedor_ruc, anio_proceso AS anio,
                            MAX(CASE WHEN proveedor_es_fantasma THEN 1 ELSE 0 END) AS f
                     FROM bm3 GROUP BY 1,2""").df()
nf = pan[pan.f==1].proveedor_ruc.nunique()
cg = pan[pan.f==1].merge(grp, on='proveedor_ruc', how='inner')
print(f'\nRUC fantasma: {nf} | con representante en el directorio: {cg.proveedor_ruc.nunique()} | '
      f'en grupo de 2+: {cg[cg.grupo_tamano>=2].proveedor_ruc.nunique()}')
dd = pan.merge(RG, left_on=['proveedor_ruc','anio'], right_on=['proveedor_ruc','anio_aplicable'], how='inner')
if dd.f.sum() > 0:
    print(f'Panel con datos de grupo: {len(dd):,} | fantasmas: {int(dd.f.sum())} filas, '
          f'{dd.loc[dd.f==1,"proveedor_ruc"].nunique()} RUC')
    print('\nMediana de percentil dentro del año (fantasma | resto):')
    for c in ['rep_proc_intragrupo_pct','rep_tasa_intragrupo_pct','grupo_tamano_pct']:
        print(f'  {c}: {dd.loc[dd.f==1,c].median():.3f} | {dd.loc[dd.f==0,c].median():.3f}')
    print(f'\nTasa intragrupo cruda -> fantasma {dd.loc[dd.f==1,"rep_tasa_intragrupo"].mean():.4f} | '
          f'resto {dd.loc[dd.f==0,"rep_tasa_intragrupo"].mean():.4f}')
else:
    print('Ningún fantasma quedó en el panel de grupos.')

In [ ]:
# ===== CAPA 2 · CONTROL DE CALIDAD + HALLAZGOS =====
import os, re, unicodedata, warnings
import numpy as np, pandas as pd, duckdb
warnings.filterwarnings('ignore')
os.chdir(r'C:\SERCOP_FINAL'); R = '03_resultados'
try: con
except NameError:
    con = duckdb.connect(); con.execute("PRAGMA threads=8")
for v, a in [('bm3','base_maestra_v3.parquet'), ('bm4','base_maestra_v4.parquet')]:
    con.execute(f"CREATE OR REPLACE VIEW {v} AS SELECT * FROM read_parquet('{R}/{a}')")
con.execute(f"CREATE OR REPLACE VIEW ofe AS SELECT * FROM read_parquet('{R}/oferentes_consolidado.parquet')")
pd.set_option('display.width', 220)

# ================= A) ¿De dónde viene el salto 2022 -> 2023? =================
print('=== A) Cobertura de oferentes por año ===')
a1 = con.execute("""
WITH p AS (SELECT ocid, anio, COUNT(DISTINCT oferente_ruc) AS n_of
           FROM ofe WHERE TRIM(COALESCE(oferente_ruc,'')) <> '' GROUP BY 1,2),
     b AS (SELECT anio_proceso AS anio, COUNT(DISTINCT ocid) AS procesos_base FROM bm3 GROUP BY 1)
SELECT b.anio, b.procesos_base,
       COUNT(p.ocid) AS procesos_con_oferentes,
       ROUND(100.0*COUNT(p.ocid)/b.procesos_base,1) AS pct_con_oferentes,
       ROUND(AVG(p.n_of),2) AS oferentes_prom,
       ROUND(AVG(CASE WHEN p.n_of >= 2 THEN 1.0 ELSE 0 END)*100,1) AS pct_2omas,
       ROUND(AVG(CASE WHEN p.n_of > 40 THEN 1.0 ELSE 0 END)*100,2) AS pct_mas_de_40
FROM b LEFT JOIN p ON p.anio = b.anio GROUP BY 1,2 ORDER BY 1""").df()
print(a1.to_string(index=False))

print('\n=== A2) ¿El ganador del proceso aparece en la lista de oferentes de SU proceso? ===')
a2 = con.execute("""
SELECT b.anio_proceso AS anio, COUNT(*) AS filas,
       ROUND(100.0*AVG(CASE WHEN o.oferente_ruc IS NOT NULL THEN 1 ELSE 0 END),1) AS pct_ganador_en_oferentes
FROM (SELECT DISTINCT ocid, proveedor_ruc, anio_proceso FROM bm3) b
LEFT JOIN (SELECT DISTINCT ocid, oferente_ruc FROM ofe) o
  ON b.ocid = o.ocid AND b.proveedor_ruc = o.oferente_ruc
GROUP BY 1 ORDER BY 1""").df()
print(a2.to_string(index=False))

print('\n=== A3) Acumulación de rivales recurrentes por año ===')
cb = pd.read_parquet(f'{R}/red_cobid_anio.parquet')[['proveedor_ruc','anio_aplicable','cob_rivales_recurrentes']]
a3 = con.execute("""
SELECT b.anio_proceso AS anio, COUNT(*) AS filas,
       ROUND(100.0*AVG(CASE WHEN c.cob_rivales_recurrentes IS NULL THEN 1 ELSE 0 END),1) AS pct_sin_historial,
       ROUND(100.0*AVG(CASE WHEN c.cob_rivales_recurrentes >= 2 THEN 1 ELSE 0 END),1) AS pct_con_2omas_recurrentes
FROM bm3 b LEFT JOIN cb c ON b.proveedor_ruc=c.proveedor_ruc AND b.anio_proceso=c.anio_aplicable
GROUP BY 1 ORDER BY 1""").df()
print(a3.to_string(index=False))

# ================= B) Representantes con muchas empresas =================
print('\n\n=== B) Representantes legales con muchas empresas en el registro ===')
dr = con.execute(f"""SELECT RUC, NOMBRE, REPRESENTANTE, CARGO, PROVINCIA
                     FROM read_parquet('{R}/directorio_companias.parquet')""").df().drop_duplicates('RUC')
def norm(s):
    if pd.isna(s): return None
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii','ignore').decode('ascii').upper()
    s = re.sub(r'[^A-Z ]',' ',s); s = re.sub(r'\s+',' ',s).strip()
    return s if len(s) >= 8 and len(s.split()) >= 2 else None
dr['rep_norm'] = dr.REPRESENTANTE.map(norm)
dr = dr[dr.rep_norm.notna()]
con.register('dr_t', dr[['RUC','rep_norm','NOMBRE']])
top = con.execute("""
WITH g AS (SELECT rep_norm, COUNT(*) AS empresas FROM dr_t GROUP BY 1),
     s AS (SELECT d.rep_norm,
                  COUNT(DISTINCT b.proveedor_ruc) AS empresas_proveedoras,
                  COUNT(DISTINCT b.ocid) AS procesos,
                  SUM(b.monto_final) AS monto
           FROM dr_t d JOIN bm3 b ON b.proveedor_ruc = d.RUC GROUP BY 1)
SELECT g.rep_norm, g.empresas,
       COALESCE(s.empresas_proveedoras,0) AS en_sercop,
       COALESCE(s.procesos,0) AS procesos,
       ROUND(COALESCE(s.monto,0)/1e6,2) AS monto_millones
FROM g LEFT JOIN s ON s.rep_norm = g.rep_norm
WHERE g.empresas >= 20 ORDER BY monto_millones DESC LIMIT 15""").df()
print(top.to_string(index=False))
tot = con.execute("""
WITH g AS (SELECT rep_norm, COUNT(*) AS e FROM dr_t GROUP BY 1 HAVING COUNT(*) >= 20)
SELECT COUNT(DISTINCT g.rep_norm) AS representantes, COUNT(DISTINCT b.proveedor_ruc) AS empresas_proveedoras,
       COUNT(DISTINCT b.ocid) AS procesos, ROUND(SUM(b.monto_final)/1e6,1) AS monto_millones
FROM g JOIN dr_t d ON d.rep_norm=g.rep_norm JOIN bm3 b ON b.proveedor_ruc=d.RUC""").df()
print('\nAgregado de representantes con 20+ empresas registradas:')
print(tot.to_string(index=False))

# ================= C) Los procesos con competencia intragrupo =================
print('\n\n=== C) Procesos donde 2+ empresas del mismo representante compitieron ===')
of = con.execute("""SELECT DISTINCT ocid, oferente_ruc AS ruc, anio FROM ofe
                    WHERE TRIM(COALESCE(oferente_ruc,'')) <> '' AND anio IS NOT NULL""").df()
t = of.groupby('ocid')['ruc'].transform('size'); of = of[(t>=2)&(t<=40)]
of = of.merge(dr[['RUC','rep_norm']], left_on='ruc', right_on='RUC', how='inner')
gs = dr.groupby('rep_norm')['RUC'].size()
of = of[of.rep_norm.map(gs).between(2,30)]
cnt = of.groupby(['ocid','rep_norm'])['ruc'].nunique()
casos = cnt[cnt>=2].reset_index().rename(columns={'ruc':'empresas_del_grupo'})
print(f'Procesos afectados: {casos.ocid.nunique():,} | representantes involucrados: {casos.rep_norm.nunique():,}')
con.register('casos_t', casos)
res = con.execute("""
SELECT b.anio_proceso AS anio, COUNT(DISTINCT b.ocid) AS procesos,
       ROUND(SUM(DISTINCT b.monto_final)/1e6,2) AS monto_millones
FROM casos_t c JOIN bm3 b ON b.ocid = c.ocid GROUP BY 1 ORDER BY 1""").df()
print('\nPor año:'); print(res.to_string(index=False))
det = con.execute("""
SELECT c.rep_norm, COUNT(DISTINCT c.ocid) AS procesos, c.empresas_del_grupo,
       ROUND(SUM(DISTINCT b.monto_final)/1e6,3) AS monto_millones
FROM casos_t c JOIN bm3 b ON b.ocid=c.ocid GROUP BY 1,3 ORDER BY monto_millones DESC LIMIT 12""").df()
print('\nCasos más grandes (candidatos a estudio de caso):'); print(det.to_string(index=False))
gan = con.execute("""
SELECT ROUND(100.0*AVG(CASE WHEN d.rep_norm IS NOT NULL THEN 1 ELSE 0 END),1) AS pct_gano_el_grupo
FROM casos_t c JOIN (SELECT DISTINCT ocid, proveedor_ruc FROM bm3) b ON b.ocid=c.ocid
LEFT JOIN dr_t d ON d.RUC=b.proveedor_ruc AND d.rep_norm=c.rep_norm""").df()
print('\n% de esos procesos que se adjudicaron a una empresa del mismo grupo:')
print(gan.to_string(index=False))

# ================= D) Tipo de contribuyente según el tercer dígito del RUC =================
# Convención del RUC ecuatoriano (VERIFICAR con documentación del SRI antes de citarla):
#   tercer dígito 0-5 = persona natural | 6 = institución pública | 9 = sociedad privada
print('\n\n=== D) Tipo de contribuyente por tercer dígito del RUC ===')
tp = con.execute("""
SELECT CASE WHEN SUBSTR(proveedor_ruc,3,1) IN ('0','1','2','3','4','5') THEN 'persona natural'
            WHEN SUBSTR(proveedor_ruc,3,1)='6' THEN 'institucion publica'
            WHEN SUBSTR(proveedor_ruc,3,1)='9' THEN 'sociedad privada'
            ELSE 'otro/indefinido' END AS tipo,
       COUNT(DISTINCT proveedor_ruc) AS proveedores, COUNT(*) AS filas,
       ROUND(SUM(monto_final)/1e6,1) AS monto_millones
FROM bm3 WHERE proveedor_ruc IS NOT NULL GROUP BY 1 ORDER BY monto_millones DESC""").df()
print(tp.to_string(index=False))
fa = con.execute("""
SELECT CASE WHEN SUBSTR(proveedor_ruc,3,1) IN ('0','1','2','3','4','5') THEN 'persona natural'
            WHEN SUBSTR(proveedor_ruc,3,1)='6' THEN 'institucion publica'
            WHEN SUBSTR(proveedor_ruc,3,1)='9' THEN 'sociedad privada'
            ELSE 'otro/indefinido' END AS tipo,
       COUNT(DISTINCT proveedor_ruc) AS ruc_fantasma
FROM bm3 WHERE proveedor_es_fantasma GROUP BY 1 ORDER BY 2 DESC""").df()
print('\nLos RUC fantasma, por tipo:'); print(fa.to_string(index=False))
en_dir = con.execute(f"""
SELECT COUNT(DISTINCT b.proveedor_ruc) AS fantasmas_en_directorio
FROM bm3 b JOIN (SELECT DISTINCT RUC FROM read_parquet('{R}/directorio_companias.parquet')) d
  ON d.RUC = b.proveedor_ruc WHERE b.proveedor_es_fantasma""").fetchone()[0]
print(f'Fantasmas que figuran en el directorio de la Superintendencia: {en_dir}')

In [ ]:
# ===== AUDITORÍA · el RUC anómalo, y los estudios de caso =====
import os, re, unicodedata, warnings
import numpy as np, pandas as pd, duckdb
warnings.filterwarnings('ignore')
os.chdir(r'C:\SERCOP_FINAL'); R = '03_resultados'
try: con
except NameError:
    con = duckdb.connect(); con.execute("PRAGMA threads=8")
con.execute(f"CREATE OR REPLACE VIEW bm3 AS SELECT * FROM read_parquet('{R}/base_maestra_v3.parquet')")
con.execute(f"CREATE OR REPLACE VIEW ofe AS SELECT * FROM read_parquet('{R}/oferentes_consolidado.parquet')")
pd.set_option('display.width', 230); pd.set_option('display.max_colwidth', 45)
cols = [c[0] for c in con.execute("DESCRIBE bm3").fetchall()]
c_pn = next((c for c in ['proveedor_nombre','nombre_proveedor','proveedor'] if c in cols), None)
c_en = next((c for c in ['entidad_nombre','nombre_entidad','entidad'] if c in cols), None)
c_tp = next((c for c in ['tipo_contratacion','metodo_contratacion','procurement_method','tipo_proceso'] if c in cols), None)
print('Columnas de nombre detectadas -> proveedor:', c_pn, '| entidad:', c_en, '| tipo:', c_tp)

# ---------- 1) Longitud y forma de los RUC ----------
print('\n=== Longitud de proveedor_ruc ===')
print(con.execute("""SELECT LENGTH(proveedor_ruc) AS largo, COUNT(DISTINCT proveedor_ruc) AS rucs,
                            COUNT(*) AS filas, ROUND(SUM(monto_final)/1e6,1) AS monto_millones
                     FROM bm3 GROUP BY 1 ORDER BY filas DESC""").df().to_string(index=False))

# ---------- 2) El RUC anómalo ----------
print('\n=== El/los RUC que no caen en 0-5, 6 ni 9 en el tercer dígito ===')
an = con.execute("""
SELECT proveedor_ruc, SUBSTR(proveedor_ruc,3,1) AS tercer_digito, LENGTH(proveedor_ruc) AS largo,
       COUNT(*) AS filas, COUNT(DISTINCT ocid) AS procesos,
       MIN(anio_proceso) AS desde, MAX(anio_proceso) AS hasta,
       ROUND(SUM(monto_final)/1e6,1) AS monto_millones
FROM bm3 WHERE SUBSTR(proveedor_ruc,3,1) NOT IN ('0','1','2','3','4','5','6','9')
GROUP BY 1,2,3 ORDER BY filas DESC LIMIT 10""").df()
print(an.to_string(index=False))
if len(an):
    ruc0 = an.proveedor_ruc.iloc[0]
    print(f'\nRUC analizado: [{ruc0}]  (repr: {ruc0!r})')
    sel = ', '.join([x for x in [f'"{c_pn}"' if c_pn else None, f'"{c_en}"' if c_en else None,
                                 f'"{c_tp}"' if c_tp else None] if x])
    if sel:
        print('\nNombres asociados a ese RUC (10 más frecuentes):')
        print(con.execute(f"""SELECT {sel}, COUNT(*) AS filas, ROUND(SUM(monto_final)/1e6,2) AS mill
                              FROM bm3 WHERE proveedor_ruc = ? GROUP BY ALL
                              ORDER BY filas DESC LIMIT 10""", [ruc0]).df().to_string(index=False))
    print('\nPor año:')
    print(con.execute("""SELECT anio_proceso AS anio, COUNT(*) AS filas, COUNT(DISTINCT ocid) AS procesos,
                                ROUND(SUM(monto_final)/1e6,1) AS mill
                         FROM bm3 WHERE proveedor_ruc = ? GROUP BY 1 ORDER BY 1""", [ruc0]).df().to_string(index=False))
    print('\n¿Está marcado como fantasma o aparece en oferentes?')
    print(con.execute("""SELECT MAX(CASE WHEN proveedor_es_fantasma THEN 1 ELSE 0 END) AS es_fantasma,
                                MAX(CASE WHEN proveedor_unico THEN 1 ELSE 0 END) AS alguna_vez_unico
                         FROM bm3 WHERE proveedor_ruc = ?""", [ruc0]).df().to_string(index=False))
    print('filas en oferentes:', con.execute("SELECT COUNT(*) FROM ofe WHERE oferente_ruc = ?", [ruc0]).fetchone()[0])

# ---------- 3) Los procesos intragrupo, normalizados por cobertura ----------
print('\n\n=== Los procesos intragrupo, contra la cobertura real de oferentes ===')
dr = con.execute(f"""SELECT RUC, NOMBRE, REPRESENTANTE FROM read_parquet('{R}/directorio_companias.parquet')""").df().drop_duplicates('RUC')
def norm(s):
    if pd.isna(s): return None
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii','ignore').decode('ascii').upper()
    s = re.sub(r'[^A-Z ]',' ',s); s = re.sub(r'\s+',' ',s).strip()
    return s if len(s) >= 8 and len(s.split()) >= 2 else None
dr['rep_norm'] = dr.REPRESENTANTE.map(norm); dr = dr[dr.rep_norm.notna()]
of = con.execute("""SELECT DISTINCT ocid, oferente_ruc AS ruc, anio FROM ofe
                    WHERE TRIM(COALESCE(oferente_ruc,'')) <> '' AND anio IS NOT NULL""").df()
t = of.groupby('ocid')['ruc'].transform('size'); of2 = of[(t>=2)&(t<=40)]
base_anio = of2.groupby('anio')['ocid'].nunique().rename('procesos_con_oferentes')
ofg = of2.merge(dr[['RUC','rep_norm']], left_on='ruc', right_on='RUC', how='inner')
gs = dr.groupby('rep_norm')['RUC'].size()
ofg = ofg[ofg.rep_norm.map(gs).between(2,30)]
cnt = ofg.groupby(['ocid','rep_norm'])['ruc'].nunique()
casos = cnt[cnt>=2].reset_index().rename(columns={'ruc':'empresas_del_grupo'})
ca = casos.merge(of2[['ocid','anio']].drop_duplicates(), on='ocid')
tab = (ca.groupby('anio')['ocid'].nunique().rename('procesos_intragrupo')
         .to_frame().join(base_anio))
tab['por_mil'] = (1000*tab.procesos_intragrupo/tab.procesos_con_oferentes).round(2)
print(tab.to_string())
print('\n(por_mil es la tasa comparable: casos por cada 1.000 procesos CON datos de oferentes)')

con.register('casos_t', casos)
print('\n=== Detalle del caso más grande por número de procesos ===')
sel2 = ', '.join([x for x in [f'b."{c_en}" AS entidad' if c_en else None,
                              f'b."{c_pn}" AS proveedor' if c_pn else None] if x])
if sel2:
    print(con.execute(f"""
      SELECT {sel2}, COUNT(DISTINCT b.ocid) AS procesos, ROUND(SUM(b.monto_final)/1e6,3) AS mill,
             MIN(b.anio_proceso) AS desde, MAX(b.anio_proceso) AS hasta
      FROM casos_t c JOIN bm3 b ON b.ocid=c.ocid
      WHERE c.rep_norm = 'PERSONA_OCULTA_02'
      GROUP BY ALL ORDER BY procesos DESC LIMIT 12""").df().to_string(index=False))
print('\n=== El proceso de $39,8M (representante PERSONA_OCULTA_01) ===')
fl = dr[dr.rep_norm=='PERSONA_OCULTA_01'][['RUC']]
con.register('fl_t', fl)
print(con.execute(f"""
  SELECT b.ocid, b.anio_proceso AS anio, {sel2 if sel2 else 'NULL AS x'},
         ROUND(b.monto_final/1e6,2) AS mill
  FROM bm3 b JOIN fl_t f ON f.RUC = b.proveedor_ruc
  ORDER BY b.monto_final DESC NULLS LAST LIMIT 5""").df().to_string(index=False))

In [ ]:
# ===== AUDITORÍA · cuánto contamina el nodo de RUC vacío =====
import os, shutil, warnings
import numpy as np, pandas as pd, duckdb
warnings.filterwarnings('ignore')
os.chdir(r'C:\SERCOP_FINAL'); R = '03_resultados'
try: con
except NameError:
    con = duckdb.connect(); con.execute("PRAGMA threads=8")
con.execute(f"CREATE OR REPLACE VIEW bm3 AS SELECT * FROM read_parquet('{R}/base_maestra_v3.parquet')")
pd.set_option('display.width', 200)

print('=== 1) Peso de los RUC no utilizables ===')
print(con.execute("""
SELECT CASE WHEN proveedor_ruc IS NULL THEN 'nulo'
            WHEN TRIM(proveedor_ruc) = '' THEN 'vacio'
            ELSE 'utilizable' END AS estado,
       COUNT(*) AS filas, COUNT(DISTINCT ocid) AS procesos,
       ROUND(SUM(monto_final)/1e6,1) AS monto_millones,
       ROUND(100.0*SUM(monto_final)/(SELECT SUM(monto_final) FROM bm3),2) AS pct_del_dinero
FROM bm3 GROUP BY 1 ORDER BY filas DESC""").df().to_string(index=False))
print('\nPor tipo de contratación (solo los no utilizables):')
print(con.execute("""
SELECT SUBSTR(tipo_contratacion,1,55) AS tipo, COUNT(*) AS filas, ROUND(SUM(monto_final)/1e6,1) AS mill
FROM bm3 WHERE proveedor_ruc IS NULL OR TRIM(proveedor_ruc)=''
GROUP BY 1 ORDER BY filas DESC LIMIT 8""").df().to_string(index=False))

print('\n\n=== 2) Qué posición ocupa el nodo vacío en la red que ya calculamos ===')
RP = pd.read_parquet(f'{R}/red_proveedor_anio.parquet')
esta = (RP.proveedor_ruc.astype(str).str.strip() == '').any()
print('El nodo de RUC vacío está en red_proveedor_anio:', esta)
if esta:
    v = RP[RP.proveedor_ruc.astype(str).str.strip() == '']
    otros = RP[RP.proveedor_ruc.astype(str).str.strip() != '']
    fil = []
    for _, r in v.iterrows():
        Y = r.anio_aplicable; o = otros[otros.anio_aplicable == Y]
        fil.append((int(Y), int(r.red_n_entidades), int(o.red_n_entidades.max()),
                    round(float(r.red_autovector), 4), round(float(o.red_autovector.max()), 4),
                    round(float((o.red_autovector < r.red_autovector).mean()), 4),
                    int(r.red_coreness), int(o.red_coreness.max())))
    print(pd.DataFrame(fil, columns=['anio','entidades_vacio','entidades_max_real','autovec_vacio',
                                     'autovec_max_real','percentil_del_vacio','core_vacio','core_max_real']
                       ).to_string(index=False))

print('\n\n=== 3) Respaldo de los resultados actuales para comparar después ===')
for a in ['red_proveedor_anio','red_entidad_anio','indices_red']:
    o = f'{R}/{a}.parquet'; n = f'{R}/{a}_v1_contaminado.parquet'
    if os.path.exists(o) and not os.path.exists(n):
        shutil.copy2(o, n); print(f'  respaldado {a} -> {a}_v1_contaminado.parquet')
    elif os.path.exists(n):
        print(f'  ya existía el respaldo de {a}')
print('\nListo. Ahora recalcular la Celda 13 con el filtro corregido.')

In [ ]:
a = pd.read_parquet(f'{R}/red_proveedor_anio_v1_contaminado.parquet')[['proveedor_ruc','anio_aplicable','red_autovector','red_coreness']]
b = pd.read_parquet(f'{R}/red_proveedor_anio.parquet')[['proveedor_ruc','anio_aplicable','red_autovector','red_coreness']]
m = a.merge(b, on=['proveedor_ruc','anio_aplicable'], suffixes=('_v1','_v2'))
print(f'Filas comparables: {len(m):,}')
for c in ['red_autovector','red_coreness']:
    r = m.groupby('anio_aplicable').apply(lambda g: g[c+'_v1'].corr(g[c+'_v2'], method='spearman'))
    print(f'\nCorrelación de Spearman v1 vs v2 en {c}, por año:'); print(r.round(3).to_string())

In [ ]:
# ===== REPARACIÓN · devolver a base_maestra_v3 lo que se perdió =====
import os, time, warnings
import numpy as np, pandas as pd, duckdb
warnings.filterwarnings('ignore')
os.chdir(r'C:\SERCOP_FINAL'); R = '03_resultados'
try: con
except NameError:
    con = duckdb.connect(); con.execute("PRAGMA threads=8")
con.execute(f"CREATE OR REPLACE VIEW bm3 AS SELECT * FROM read_parquet('{R}/base_maestra_v3.parquet')")
con.execute(f"CREATE OR REPLACE VIEW directorio AS SELECT * FROM read_parquet('{R}/directorio_companias.parquet')")
cols = [c[0] for c in con.execute("DESCRIBE bm3").fetchall()]
print(f'base_maestra_v3 actual: {len(cols)} columnas')

# --- 1) CPC: localizar el archivo y su columna ---
cpc_ok = os.path.exists(f'{R}/cpc_por_ocid.parquet')
if cpc_ok:
    con.execute(f"CREATE OR REPLACE VIEW cpc AS SELECT * FROM read_parquet('{R}/cpc_por_ocid.parquet')")
    cpc_cols = [c[0] for c in con.execute("DESCRIBE cpc").fetchall()]
    print('cpc_por_ocid:', cpc_cols, '| filas:', con.execute("SELECT COUNT(*) FROM cpc").fetchone()[0])
    c_ocid = next((c for c in cpc_cols if c.lower() == 'ocid'), None)
    cpc_val = [c for c in cpc_cols if c != c_ocid]
    dup = con.execute(f'SELECT COUNT(*) FROM (SELECT "{c_ocid}" FROM cpc GROUP BY 1 HAVING COUNT(*)>1)').fetchone()[0]
    print(f'  llave={c_ocid} | columnas a traer={cpc_val} | ocid duplicados={dup}')
    assert dup == 0, 'cpc_por_ocid tiene ocid duplicados: deduplicar antes'
else:
    print('!! No existe cpc_por_ocid.parquet: el CPC no se puede reponer aquí')
    cpc_val = []

# --- 2) Reconstruir, quitando lo roto y reponiendo lo perdido ---
quitar = [c for c in ['proveedor_capital_suscrito','ratio_monto_capital','entrada_anio_proceso',
                      'sin_historial_como_oferente','sin_pares_recurrentes','sin_historial_benford']
          if c in cols] + [c for c in cpc_val if c in cols]
print('\nColumnas que se reemplazan:', quitar)
sel_cpc = ', '.join([f'cpc."{c}"' for c in cpc_val]) if cpc_val else ''

sql = f"""
COPY (
  WITH cap AS (
      SELECT RUC,
             MAX(TRY_CAST(REPLACE(REPLACE("CAPITAL SUSCRITO", '.', ''), ',', '.') AS DOUBLE)) AS capital
      FROM directorio GROUP BY RUC
  )
  SELECT b.* EXCLUDE ({', '.join(quitar)}),
         cap.capital AS proveedor_capital_suscrito,
         CASE WHEN cap.capital > 0 THEN b.monto_final / cap.capital END AS ratio_monto_capital,
         (b.n_participaciones IS NULL)   AS sin_historial_como_oferente,
         (b.total_coincidencias IS NULL) AS sin_pares_recurrentes,
         (b.mad_benford IS NULL)         AS sin_historial_benford
         {(', ' + sel_cpc) if sel_cpc else ''}
  FROM bm3 b
  LEFT JOIN cap ON b.proveedor_ruc = cap.RUC
  {f'LEFT JOIN cpc ON b.ocid = cpc."{c_ocid}"' if cpc_val else ''}
) TO '{R}/base_maestra_v3.tmp.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
""" if quitar else None
assert sql is not None, 'No hay nada que reemplazar: revisar nombres'
t0 = time.time(); con.execute(sql); print(f'\nEscrita en {time.time()-t0:.0f} s')
con.execute("DROP VIEW bm3")
os.replace(f'{R}/base_maestra_v3.tmp.parquet', f'{R}/base_maestra_v3.parquet')
con.execute(f"CREATE OR REPLACE VIEW bm3 AS SELECT * FROM read_parquet('{R}/base_maestra_v3.parquet')")

n = con.execute("SELECT COUNT(*) FROM bm3").fetchone()[0]
c2 = [c[0] for c in con.execute("DESCRIBE bm3").fetchall()]
print(f'Filas: {n:,} | columnas: {len(cols)} -> {len(c2)}')
assert n == 2736387, f'El número de filas cambió: {n:,}'

# --- 3) Verificar que lo reparado quedó bien ---
print('\nCobertura de lo reparado:')
chk = ['proveedor_capital_suscrito','ratio_monto_capital'] + cpc_val
ex = ', '.join([f'ROUND(100.0*AVG(CASE WHEN "{c}" IS NOT NULL THEN 1 ELSE 0 END),1) AS "{c}"' for c in chk])
print(con.execute(f"SELECT anio_proceso AS anio, {ex} FROM bm3 GROUP BY 1 ORDER BY 1").df().to_string(index=False))
print('\nCapital suscrito (la mediana debería rondar los $800):')
print(con.execute("""SELECT COUNT(proveedor_capital_suscrito) AS con_dato,
                            ROUND(MEDIAN(proveedor_capital_suscrito),2) AS mediana,
                            ROUND(MIN(proveedor_capital_suscrito),2) AS minimo,
                            ROUND(MAX(proveedor_capital_suscrito),2) AS maximo
                     FROM bm3""").df().to_string(index=False))
print('\nBanderas de faltantes (% en TRUE):')
print(con.execute("""SELECT ROUND(100.0*AVG(CASE WHEN sin_historial_como_oferente THEN 1 ELSE 0 END),1) AS sin_oferente,
                            ROUND(100.0*AVG(CASE WHEN sin_pares_recurrentes THEN 1 ELSE 0 END),1) AS sin_pares,
                            ROUND(100.0*AVG(CASE WHEN sin_historial_benford THEN 1 ELSE 0 END),1) AS sin_benford
                     FROM bm3""").df().to_string(index=False))
print('\nListo. Ahora sí, correr la Celda 19 para regenerar base_maestra_v4.')

In [ ]:
# ===== VERIFICACIÓN del capital + RECÁLCULO de concentración sin RUC inválido =====
import os, warnings
import numpy as np, pandas as pd, duckdb
warnings.filterwarnings('ignore')
os.chdir(r'C:\SERCOP_FINAL'); R = '03_resultados'
try: con
except NameError:
    con = duckdb.connect(); con.execute("PRAGMA threads=8")
con.execute(f"CREATE OR REPLACE VIEW bm3 AS SELECT * FROM read_parquet('{R}/base_maestra_v3.parquet')")
con.execute(f"CREATE OR REPLACE VIEW directorio AS SELECT * FROM read_parquet('{R}/directorio_companias.parquet')")
# vista canónica: solo RUC utilizables. USARLA en todo análisis por proveedor.
con.execute("""CREATE OR REPLACE VIEW bmv AS SELECT * FROM bm3
               WHERE proveedor_ruc IS NOT NULL AND TRIM(proveedor_ruc) <> ''""")
pd.set_option('display.width', 200)

print('=== 1) ¿El formato del capital se está leyendo bien? ===')
print(con.execute("""
SELECT "CAPITAL SUSCRITO" AS texto_original,
       TRY_CAST(REPLACE(REPLACE("CAPITAL SUSCRITO",'.',''),',','.') AS DOUBLE) AS leido_latino,
       TRY_CAST("CAPITAL SUSCRITO" AS DOUBLE) AS leido_directo, COUNT(*) AS empresas
FROM directorio GROUP BY 1,2,3 ORDER BY empresas DESC LIMIT 12""").df().to_string(index=False))
print('\nFormato de los textos de capital:')
print(con.execute("""
SELECT CASE WHEN "CAPITAL SUSCRITO" LIKE '%,%' AND "CAPITAL SUSCRITO" LIKE '%.%' THEN 'tiene punto Y coma'
            WHEN "CAPITAL SUSCRITO" LIKE '%,%' THEN 'solo coma'
            WHEN "CAPITAL SUSCRITO" LIKE '%.%' THEN 'solo punto'
            ELSE 'sin separadores' END AS forma, COUNT(*) AS empresas
FROM directorio GROUP BY 1 ORDER BY empresas DESC""").df().to_string(index=False))
print('\nCapital POR EMPRESA (no por fila): aquí debería aparecer el mínimo legal')
print(con.execute("""
WITH e AS (SELECT DISTINCT proveedor_ruc, proveedor_capital_suscrito AS k FROM bmv
           WHERE proveedor_capital_suscrito IS NOT NULL)
SELECT COUNT(*) AS empresas, ROUND(MEDIAN(k),2) AS mediana,
       ROUND(QUANTILE_CONT(k,0.25),2) AS p25, ROUND(QUANTILE_CONT(k,0.75),2) AS p75,
       SUM(CASE WHEN k = 800 THEN 1 ELSE 0 END) AS exactamente_800,
       SUM(CASE WHEN k <= 1000 THEN 1 ELSE 0 END) AS hasta_1000 FROM e""").df().to_string(index=False))

print('\n\n=== 2) Concentración del gasto, SIN el RUC vacío ni nulo ===')
print(con.execute("""
WITH p AS (SELECT proveedor_ruc, SUM(monto_final) AS m FROM bmv
           WHERE monto_final IS NOT NULL GROUP BY 1),
     r AS (SELECT m, PERCENT_RANK() OVER (ORDER BY m DESC) AS pr FROM p)
SELECT (SELECT COUNT(*) FROM p) AS proveedores,
       ROUND((SELECT SUM(m) FROM p)/1e6,1) AS monto_total_millones,
       ROUND(100.0*SUM(CASE WHEN pr < 0.01 THEN m ELSE 0 END)/SUM(m),1) AS pct_top1,
       ROUND(100.0*SUM(CASE WHEN pr < 0.10 THEN m ELSE 0 END)/SUM(m),1) AS pct_top10,
       ROUND(100.0*SUM(CASE WHEN pr >= 0.50 THEN m ELSE 0 END)/SUM(m),1) AS pct_mitad_inferior
FROM r""").df().to_string(index=False))
print('\nComparación: la misma cifra INCLUYENDO el RUC vacío (lo que decía el documento)')
print(con.execute("""
WITH p AS (SELECT COALESCE(proveedor_ruc,'(nulo)') AS ruc, SUM(monto_final) AS m FROM bm3
           WHERE monto_final IS NOT NULL GROUP BY 1),
     r AS (SELECT m, PERCENT_RANK() OVER (ORDER BY m DESC) AS pr FROM p)
SELECT ROUND(100.0*SUM(CASE WHEN pr < 0.01 THEN m ELSE 0 END)/SUM(m),1) AS pct_top1,
       ROUND(100.0*SUM(CASE WHEN pr < 0.10 THEN m ELSE 0 END)/SUM(m),1) AS pct_top10
FROM r""").df().to_string(index=False))

print('\n\n=== 3) Monto contratado muy por encima del capital, recalculado ===')
print(con.execute("""
SELECT COUNT(*) AS procesos, COUNT(DISTINCT proveedor_ruc) AS empresas,
       ROUND(SUM(monto_final)/1e6,1) AS monto_millones,
       ROUND(MEDIAN(proveedor_capital_suscrito),2) AS capital_mediano
FROM bmv WHERE ratio_monto_capital > 100""").df().to_string(index=False))

print('\n\n=== 4) Dinero por tipo de contribuyente, sin los RUC inválidos ===')
print(con.execute("""
SELECT CASE WHEN SUBSTR(proveedor_ruc,3,1) IN ('0','1','2','3','4','5') THEN 'persona natural'
            WHEN SUBSTR(proveedor_ruc,3,1)='6' THEN 'institucion publica'
            WHEN SUBSTR(proveedor_ruc,3,1)='9' THEN 'sociedad privada'
            ELSE 'otro' END AS tipo,
       COUNT(DISTINCT proveedor_ruc) AS proveedores, COUNT(*) AS filas,
       ROUND(SUM(monto_final)/1e6,1) AS monto_millones,
       ROUND(100.0*SUM(monto_final)/(SELECT SUM(monto_final) FROM bmv),1) AS pct_dinero
FROM bmv GROUP BY 1 ORDER BY monto_millones DESC""").df().to_string(index=False))
print('\n(bmv es la vista canónica de aquí en adelante: excluye RUC vacío y nulo)')

In [ ]:
# ===== CIERRE FINAL · base_maestra_v4 (correr DESPUÉS de la reparación) =====
import os, time, warnings
import numpy as np, pandas as pd, duckdb
warnings.filterwarnings('ignore')
os.chdir(r'C:\SERCOP_FINAL'); R = '03_resultados'
try: con
except NameError:
    con = duckdb.connect(); con.execute("PRAGMA threads=8")
con.execute(f"CREATE OR REPLACE VIEW bm3 AS SELECT * FROM read_parquet('{R}/base_maestra_v3.parquet')")
print('v3 de entrada:', len(con.execute("DESCRIBE bm3").fetchall()), 'columnas (deben ser 110)')

SAT = {
 'rp':  ('red_proveedor_anio.parquet',    'proveedor_ruc', 'anio_aplicable'),
 'rc':  ('red_cobid_anio.parquet',        'proveedor_ruc', 'anio_aplicable'),
 'rm':  ('red_comunidad_anio.parquet',    'proveedor_ruc', 'anio_aplicable'),
 'rr':  ('red_representante_anio.parquet','proveedor_ruc', 'anio_aplicable'),
 'ix':  ('indices_red.parquet',           'proveedor_ruc', 'anio'),
 're':  ('red_entidad_anio.parquet',      'entidad_ruc',   'anio_aplicable'),
}
usar, faltan = {}, []
for a, (arch, k, ka) in SAT.items():
    ruta = f'{R}/{arch}'
    if not os.path.exists(ruta):
        faltan.append(arch); continue
    con.execute(f"CREATE OR REPLACE VIEW v_{a} AS SELECT * FROM read_parquet('{ruta}')")
    dup = con.execute(f"SELECT COUNT(*) FROM (SELECT {k},{ka} FROM v_{a} GROUP BY 1,2 HAVING COUNT(*)>1)").fetchone()[0]
    n = con.execute(f"SELECT COUNT(*) FROM v_{a}").fetchone()[0]
    cols = [c[0] for c in con.execute(f"DESCRIBE v_{a}").fetchall()]
    print(f'[{a}] {arch}: {n:,} filas | {len(cols)} cols | llaves duplicadas: {dup}')
    assert dup == 0, f'{arch} tiene llaves duplicadas por ({k},{ka})'
    usar[a] = (arch, k, ka, cols)
if faltan: print('\n!! No encontrados (se omiten):', faltan)

bm_cols = [c[0] for c in con.execute("DESCRIBE bm3").fetchall()]
ya = set(bm_cols); sel, joins = [], []
for a, (arch, k, ka, cols) in usar.items():
    for c in [c for c in cols if c not in (k, ka)]:
        nombre = c if c not in ya else f'{a}_{c}'
        sel.append(f'{a}."{c}" AS "{nombre}"'); ya.add(nombre)
    joins.append(f'LEFT JOIN v_{a} {a} ON b."{k}" = {a}."{k}" AND b.anio_proceso = {a}."{ka}"')
print(f'\nColumnas nuevas a agregar: {len(sel)}')

sql = f"""COPY (SELECT b.*, {', '.join(sel)} FROM bm3 b {' '.join(joins)})
          TO '{R}/base_maestra_v4.tmp.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)"""
t0 = time.time(); con.execute(sql); print(f'escrita en {time.time()-t0:.0f} s')
try: con.execute("DROP VIEW bm4")
except Exception: pass
os.replace(f'{R}/base_maestra_v4.tmp.parquet', f'{R}/base_maestra_v4.parquet')
con.execute(f"CREATE OR REPLACE VIEW bm4 AS SELECT * FROM read_parquet('{R}/base_maestra_v4.parquet')")

n3 = con.execute("SELECT COUNT(*) FROM bm3").fetchone()[0]
n4 = con.execute("SELECT COUNT(*) FROM bm4").fetchone()[0]
c4 = [c[0] for c in con.execute("DESCRIBE bm4").fetchall()]
print(f'Filas v3: {n3:,} | v4: {n4:,} | columnas: {len(bm_cols)} -> {len(c4)}')
assert n3 == n4, 'Algún join duplicó filas'

# verificación de que lo reparado llegó a la v4
print('\nLo reparado, presente en v4:')
for c in ['proveedor_capital_suscrito','ratio_monto_capital','cpc5_principal',
          'sin_historial_como_oferente','sin_pares_recurrentes','sin_historial_benford']:
    print(f'  {"OK " if c in c4 else "FALTA"} {c}')
print('\nCobertura del capital en v4:',
      con.execute("""SELECT ROUND(100.0*AVG(CASE WHEN proveedor_capital_suscrito IS NOT NULL THEN 1 ELSE 0 END),1)
                     FROM bm4 WHERE TRIM(COALESCE(proveedor_ruc,'')) <> ''""").fetchone()[0], '%')
print('\nbase_maestra_v4 lista. Esta es la entrada del modelo de la Capa 3.')